# Experimento VQE — versão 19.5.5
## Máscaras fixas e cronogramas adaptativos: 7 → 30 → 7 parâmetros

Esta versão mantém toda a reconstrução clássica, a codificação QUBO/Ising, o ansatz de Dicke e os bancos para o Transformer. A extensão principal substitui a comparação isolada `active_7` versus `full_30` por um experimento pareado com oito regimes.

O teste central solicitado é:

\[
7\text{ ativos por }20\text{ ou }30\text{ avaliações}
\rightarrow
30\text{ parâmetros até a avaliação }150
\rightarrow
7\text{ ativos até a avaliação }250.
\]

A hipótese operacional é que os sete parâmetros podem acelerar a descida inicial, os 30 parâmetros podem transportar o circuito até a bacia ótima, e os sete podem realizar o refinamento final com menor dimensionalidade.


## O que foi alterado e o que foi preservado

### Preservado

- reconstrução clássica exata para o problema de portfólio;
- auditoria histórica do caso $k=4$;
- Hamiltoniano QUBO/Ising;
- ansatz de Dicke com 30 parâmetros;
- `BackendEstimatorV2` com `AerSimulator` dentro do VQE;
- avaliação final exata com `Statevector`;
- distribuição final com 4096 shots;
- bancos multi-$k$ para o Transformer.

### Novo experimento

São comparados, com os mesmos pontos iniciais e o mesmo orçamento nominal de 250 avaliações:

1. `full_30`: 30 parâmetros durante toda a busca;
2. `active7_random`: apenas os sete, com os outros 23 congelados no ponto aleatório inicial;
3. `active7_zero`: apenas os sete, com os outros 23 fixados em zero;
4. `active7_anchor`: apenas os sete, com os outros 23 fixados numa âncora previamente otimizada;
5. `hybrid_7x20_30to150_7to250`;
6. `hybrid_7x30_30to150_7to250`;
7. `hybrid_30to150_7to250`, que mede se o aquecimento inicial com sete é realmente útil;
8. `hybrid_7x30_30to250`, que mede se o retorno final aos sete melhora o resultado.

Cada mudança de máscara reinicia apenas o estado interno do COBYLA. O vetor físico de parâmetros é transmitido continuamente de uma etapa para a seguinte.


## Hipóteses pré-registradas

- **H1 — suficiência local:** os sete índices $\{2,14,17,19,22,25,27\}$ controlam o refinamento dentro da bacia ótima.
- **H2 — navegação global:** parte dos outros 23 parâmetros é necessária para transportar o circuito de uma inicialização arbitrária até essa bacia.
- **H3 — fundo estrutural:** `active7_anchor` deve superar `active7_random` caso os 23 parâmetros possam ser tratados como constantes estruturais, mas não como valores arbitrários.
- **H4 — aquecimento reduzido:** 20 ou 30 avaliações apenas nos sete podem produzir uma descida inicial barata antes de liberar todos os parâmetros.
- **H5 — refinamento final:** após a fase completa até a avaliação 150, congelar novamente os 23 deve preservar ou melhorar a qualidade final com menor dimensionalidade.
- **H6 — ablação do cronograma:** comparar `hybrid_7x30_30to150_7to250`, `hybrid_30to150_7to250` e `hybrid_7x30_30to250` separa o efeito do aquecimento inicial, da fase global e do refinamento final.

O teste usa `nfev`, isto é, número de avaliações da função objetivo, como unidade de interação entre o COBYLA e o estimador quântico.


In [ ]:
# ============================================================
# 1. IMPORTS, CAMINHOS E CONFIGURAÇÃO ÚNICA
# ============================================================

from __future__ import annotations

from itertools import combinations
from pathlib import Path
from time import perf_counter
import hashlib
import json
import math
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr, wilcoxon

NOTEBOOK_VERSION = "19.5.5"

# Problema original
N_ASSETS = 10
TARGET_K = 4
K_VALUES = (2, 3, 4, 5)
Q_VALUE = 0.5
RISK_FREE = 0.0475
RANDOM_SEED = 42

# Auditoria histórica do caso k=4. O bitstring está na ordem exibida pelo Qiskit.
KNOWN_EXACT_QISKIT_BITSTRING = "1001011000"
KNOWN_EXACT_ENERGY = -0.8181062577496281
STRICT_HISTORICAL_AUDIT = True

# Critérios definidos antes de olhar os rankings.
ENERGY_EQUALITY_ATOL = 1e-8
COMPETITIVE_REL_TOL = 0.01

# Banco principal.
BANK_RUNS_PER_CONFIGURATION = 100
COBYLA_MAXITER = 250
SHOTS = 4096
ESTIMATOR_PRECISION = float(1.0 / np.sqrt(SHOTS))
GENERATE_OR_RESUME_BANKS = False

# Experimento de máscaras e cronogramas.
RUN_STAGED_MASK_EXPERIMENT = False
ACTIVE_THETA_INDICES = (2, 14, 17, 19, 22, 25, 27)
STAGED_EXPERIMENT_RESTARTS = 100
STAGED_TOTAL_BUDGET = 250
STAGED_FULL_PHASE_END = 150
STAGED_ACTIVE_WARMUP_BUDGETS = (20, 30)
STAGED_DISTRIBUTION_SHOTS = SHOTS
STAGED_SUCCESS_P_THRESHOLD = 0.99
STAGED_SUCCESS_GAP_ATOL = 1e-3
STAGED_STRICT_GAP_ATOL = 1e-6
STAGED_ESTIMATOR_GAP_THRESHOLDS = (1e-2, 1e-3, 1e-4)

# A âncora é procurada primeiro nos bancos já existentes. Se nenhuma solução
# estiver disponível, estes restarts independentes constroem uma referência.
ANCHOR_CALIBRATION_RESTARTS = 5
ANCHOR_CALIBRATION_MAXITER = COBYLA_MAXITER


def locate_project_file(relative_candidates, max_parent_levels=6):
    """Procura um arquivo a partir da pasta atual e das pastas superiores."""
    roots = [Path.cwd().resolve(), *list(Path.cwd().resolve().parents)[:max_parent_levels]]
    checked = []
    for candidate in relative_candidates:
        candidate = Path(candidate).expanduser()
        candidates = [candidate.resolve()] if candidate.is_absolute() else [
            (root / candidate).resolve() for root in roots
        ]
        for path in candidates:
            if path not in checked:
                checked.append(path)
            if path.is_file():
                return path, checked
    return None, checked


custom_csv = os.environ.get("VQE_STOCKS_CSV")
stock_candidates = [
    *([Path(custom_csv)] if custom_csv else []),
    Path("data/assets/stocks_price.csv"),
    Path("data/assets/stocks_prices.csv"),
    Path("../data/assets/stocks_price.csv"),
    Path("../data/assets/stocks_prices.csv"),
]
stock_path, checked_paths = locate_project_file(stock_candidates)

if stock_path is None:
    checked_text = "\n".join(f" - {path}" for path in checked_paths)
    raise FileNotFoundError(
        "O CSV de preços não foi localizado.\n"
        "Defina VQE_STOCKS_CSV ou coloque o arquivo em data/assets/.\n"
        f"Caminhos verificados:\n{checked_text}"
    )

results_root = Path(os.environ.get("VQE_RESULTS_DIR", "vqe_r"))
output_dir = results_root / "pipeline_v19_5"
classical_dir = output_dir / "01_classical"
quantum_dir = output_dir / "02_quantum"
bank_dir = output_dir / "03_banks"
transformer_dir = output_dir / "04_transformer_dataset"
legacy_active7_dir = output_dir / "05_active7_vs_full30_experiment"
staged_dir = output_dir / "06_staged_mask_experiment"
figure_dir = output_dir / "figures"
for directory in [
    output_dir,
    classical_dir,
    quantum_dir,
    bank_dir,
    transformer_dir,
    staged_dir,
    figure_dir,
]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "notebook_version": NOTEBOOK_VERSION,
    "n_assets": N_ASSETS,
    "target_k": TARGET_K,
    "k_values": list(K_VALUES),
    "q_value": Q_VALUE,
    "risk_free": RISK_FREE,
    "random_seed": RANDOM_SEED,
    "strict_historical_audit": STRICT_HISTORICAL_AUDIT,
    "energy_equality_atol": ENERGY_EQUALITY_ATOL,
    "competitive_relative_tolerance": COMPETITIVE_REL_TOL,
    "bank_runs_per_configuration": BANK_RUNS_PER_CONFIGURATION,
    "cobyla_maxiter": COBYLA_MAXITER,
    "shots": SHOTS,
    "estimator_precision": ESTIMATOR_PRECISION,
    "generate_or_resume_banks": GENERATE_OR_RESUME_BANKS,
    "run_staged_mask_experiment": RUN_STAGED_MASK_EXPERIMENT,
    "active_theta_indices": list(ACTIVE_THETA_INDICES),
    "staged_experiment_restarts": STAGED_EXPERIMENT_RESTARTS,
    "staged_total_budget": STAGED_TOTAL_BUDGET,
    "staged_full_phase_end": STAGED_FULL_PHASE_END,
    "staged_active_warmup_budgets": list(STAGED_ACTIVE_WARMUP_BUDGETS),
    "staged_distribution_shots": STAGED_DISTRIBUTION_SHOTS,
    "staged_success_p_threshold": STAGED_SUCCESS_P_THRESHOLD,
    "staged_success_gap_atol": STAGED_SUCCESS_GAP_ATOL,
    "staged_strict_gap_atol": STAGED_STRICT_GAP_ATOL,
    "staged_estimator_gap_thresholds": list(STAGED_ESTIMATOR_GAP_THRESHOLDS),
    "anchor_calibration_restarts": ANCHOR_CALIBRATION_RESTARTS,
    "anchor_calibration_maxiter": ANCHOR_CALIBRATION_MAXITER,
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print("CSV:", stock_path.resolve())
print("saídas:", output_dir.resolve())


# Parte I — reconstrução clássica exata

Esta parte não usa circuito quântico. Ela define a paisagem combinatória que qualquer método quântico ou Transformer deverá reproduzir.

A função objetivo preserva exatamente as escolhas do experimento original:

\[
E(x)=q\,x^T\Sigma x-(1-q)\,\mu^Tx+r_f,
\qquad \sum_i x_i=k.
\]

O retorno usado é a **soma dos retornos diários**, não a média. Alterar essa escolha cria outro Hamiltoniano.


In [ ]:
# ============================================================
# 2. PREÇOS, RETORNOS E COVARIÂNCIA
# ============================================================

stocks_prices = pd.read_csv(stock_path, index_col=0)
stocks_prices = (
    stocks_prices
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .dropna(axis=1, how="all")
    .ffill()
    .bfill()
    .dropna()
)

if stocks_prices.empty:
    raise ValueError("O CSV não contém dados numéricos válidos.")
if stocks_prices.columns.duplicated().any():
    raise ValueError("O CSV possui tickers repetidos.")
if stocks_prices.shape[1] != N_ASSETS:
    raise ValueError(
        f"Esperados {N_ASSETS} ativos; encontrados {stocks_prices.shape[1]}: "
        f"{stocks_prices.columns.tolist()}"
    )
if len(stocks_prices) < 3:
    raise ValueError("São necessárias ao menos três datas.")

daily_returns = stocks_prices.pct_change(fill_method=None).dropna()
assets_returns = daily_returns.sum()
covariance = daily_returns.cov()

mu = assets_returns.to_numpy(dtype=float)
sigma = covariance.to_numpy(dtype=float)
sigma = 0.5 * (sigma + sigma.T)
tickers = list(stocks_prices.columns)

if not np.all(np.isfinite(mu)) or not np.all(np.isfinite(sigma)):
    raise ValueError("Retornos ou covariância contêm valores não finitos.")

min_cov_eigenvalue = float(np.linalg.eigvalsh(sigma).min())
if min_cov_eigenvalue < -1e-10:
    raise ValueError(
        "A covariância não é semidefinida positiva dentro da tolerância: "
        f"lambda_min={min_cov_eigenvalue:.3e}"
    )

asset_summary_df = pd.DataFrame({
    "asset_index": np.arange(N_ASSETS, dtype=int),
    "ticker": tickers,
    "return_sum": mu,
    "variance": np.diag(sigma),
    "risk_connection_abs": np.sum(np.abs(sigma), axis=1),
})

DATA_HASH = hashlib.sha256(
    np.concatenate([mu, sigma.reshape(-1)]).astype(np.float64).tobytes()
).hexdigest()[:16]

print("preços:", stocks_prices.shape)
print("retornos diários:", daily_returns.shape)
print("tickers:", tickers)
print("data_hash:", DATA_HASH)
print("menor autovalor da covariância:", f"{min_cov_eigenvalue:.3e}")
display(asset_summary_df)


## 3. Funções exatas compartilhadas por todos os valores de $k$

A mesma infraestrutura é usada no caso original $k=4$ e, posteriormente, nos contextos multi-$k$. Isso evita manter versões diferentes da função objetivo ou do cálculo de $G_{ij}$.

Para cada par $(i,j)$ são calculados os quatro mínimos condicionados:

\[
E_{ij}^{ab}=\min_{x_i=a,x_j=b,\,\sum x=k}E(x),\qquad ab\in\{00,01,10,11\}.
\]

O gap principal é

\[
G_{ij}=E_{ij}^{\text{segundo estado}}-E^*.
\]

Também são preservadas as quatro penalidades direcionais, a inversão conjunta, a não aditividade e o número de trocas necessárias para sair do ótimo.


In [ ]:
# ============================================================
# 3. ENUMERAÇÃO, MARGEM ANTIGA E GAPS CONDICIONAIS
# ============================================================

PAIR_STATES = ("00", "01", "10", "11")


def portfolio_objective(x_binary, mu_values=mu, sigma_values=sigma):
    x = np.asarray(x_binary, dtype=float).reshape(-1)
    return float(
        Q_VALUE * x @ sigma_values @ x
        - (1.0 - Q_VALUE) * x @ mu_values
        + RISK_FREE
    )


def bitstring_from_x(x):
    return "".join(str(int(value)) for value in np.asarray(x, dtype=int))


def selected_assets_from_x(x, labels=tickers):
    return [ticker for ticker, selected in zip(labels, x) if int(selected) == 1]


def enumerate_portfolios(k_value, mu_values=mu, sigma_values=sigma, labels=tickers):
    n_value = len(labels)
    rows = []
    for selected_indices in combinations(range(n_value), int(k_value)):
        x = np.zeros(n_value, dtype=int)
        x[list(selected_indices)] = 1
        bits = bitstring_from_x(x)
        rows.append({
            "selected_indices": tuple(int(i) for i in selected_indices),
            "x_asset_order": tuple(int(v) for v in x),
            "bitstring_asset_order": bits,
            "bitstring_qiskit_order": bits[::-1],
            "selected_assets": tuple(selected_assets_from_x(x, labels)),
            "objective": portfolio_objective(x, mu_values, sigma_values),
        })
    return (
        pd.DataFrame(rows)
        .sort_values(["objective", "bitstring_asset_order"])
        .reset_index(drop=True)
    )


def old_asset_decision_margins(exact_x, exact_energy, energy_span, labels=tickers):
    selected_indices = np.flatnonzero(exact_x == 1)
    excluded_indices = np.flatnonzero(exact_x == 0)
    rows = []
    for asset_index in range(len(labels)):
        alternatives = []
        if exact_x[asset_index] == 1:
            for replacement in excluded_indices:
                trial = exact_x.copy()
                trial[asset_index] = 0
                trial[replacement] = 1
                alternatives.append((portfolio_objective(trial), trial))
        else:
            for removed in selected_indices:
                trial = exact_x.copy()
                trial[asset_index] = 1
                trial[removed] = 0
                alternatives.append((portfolio_objective(trial), trial))
        best_energy, best_trial = min(alternatives, key=lambda item: item[0])
        margin = max(float(best_energy - exact_energy), 0.0)
        rows.append({
            "asset_index": int(asset_index),
            "ticker": labels[asset_index],
            "selected_exact": int(exact_x[asset_index]),
            "decision_margin": margin,
            "decision_margin_relative": margin / max(energy_span, ENERGY_EQUALITY_ATOL),
            "best_flip_bitstring": bitstring_from_x(best_trial),
            "best_flip_selected_assets": tuple(selected_assets_from_x(best_trial, labels)),
            "return_coefficient": float(-(1.0 - Q_VALUE) * mu[asset_index]),
            "self_risk_coefficient": float(Q_VALUE * sigma[asset_index, asset_index]),
            "risk_connection": float(Q_VALUE * np.sum(np.abs(sigma[asset_index]))),
        })
    return pd.DataFrame(rows).sort_values("decision_margin", ascending=False).reset_index(drop=True)


def build_pair_gap_table(enumeration_df, exact_x, exact_energy, energy_span, asset_margin_df, labels=tickers):
    n_value = len(labels)
    margin_lookup = asset_margin_df.set_index("asset_index")["decision_margin"].to_dict()

    def conditional_best(i, j, state):
        a, b = int(state[0]), int(state[1])
        mask = enumeration_df["x_asset_order"].map(
            lambda x: int(x[i]) == a and int(x[j]) == b
        )
        subset = enumeration_df.loc[mask]
        if subset.empty:
            raise RuntimeError(f"Estado inviável para par {(i, j)}: {state}")
        best = subset.iloc[0]
        return {
            "count": int(len(subset)),
            "energy": float(best["objective"]),
            "bitstring": str(best["bitstring_asset_order"]),
            "selected_assets": tuple(best["selected_assets"]),
        }

    rows = []
    for i, j in combinations(range(n_value), 2):
        state_results = {state: conditional_best(i, j, state) for state in PAIR_STATES}
        ordered_states = sorted(PAIR_STATES, key=lambda s: (state_results[s]["energy"], s))
        best_state, second_state = ordered_states[:2]
        best_energy = state_results[best_state]["energy"]
        second_energy = state_results[second_state]["energy"]
        gij = max(float(second_energy - best_energy), 0.0)

        states_at_best = tuple(
            state for state in PAIR_STATES
            if np.isclose(state_results[state]["energy"], best_energy, atol=ENERGY_EQUALITY_ATOL, rtol=0.0)
        )
        second_x = np.asarray([int(v) for v in state_results[second_state]["bitstring"]], dtype=int)
        hamming = int(np.sum(second_x != exact_x))
        removed = [labels[idx] for idx in np.flatnonzero((exact_x == 1) & (second_x == 0))]
        added = [labels[idx] for idx in np.flatnonzero((exact_x == 0) & (second_x == 1))]
        n_swaps = hamming // 2

        global_pair_state = f"{exact_x[i]}{exact_x[j]}"
        decision_pattern = {
            "00": "exclude_exclude",
            "01": "exclude_include",
            "10": "include_exclude",
            "11": "include_include",
        }[global_pair_state]
        a_star, b_star = int(global_pair_state[0]), int(global_pair_state[1])
        single_i_state = f"{1-a_star}{b_star}"
        single_j_state = f"{a_star}{1-b_star}"
        joint_state = f"{1-a_star}{1-b_star}"
        single_i_gap = max(state_results[single_i_state]["energy"] - exact_energy, 0.0)
        single_j_gap = max(state_results[single_j_state]["energy"] - exact_energy, 0.0)
        joint_gap = max(state_results[joint_state]["energy"] - exact_energy, 0.0)
        nonadditivity = joint_gap - single_i_gap - single_j_gap
        exit_gap = min(single_i_gap, single_j_gap, joint_gap)
        delta_01_10 = abs(state_results["01"]["energy"] - state_results["10"]["energy"])
        top2_swap = set(ordered_states[:2]) == {"01", "10"}

        rows.append({
            "asset_i": labels[i], "asset_j": labels[j],
            "asset_index_i": int(i), "asset_index_j": int(j),
            "pair": f"{labels[i]}/{labels[j]}",
            "global_pair_state": global_pair_state,
            "decision_pattern": decision_pattern,
            "single_flip_i_state": single_i_state,
            "single_flip_j_state": single_j_state,
            "joint_complement_state": joint_state,
            "single_flip_i_gap": float(single_i_gap),
            "single_flip_j_gap": float(single_j_gap),
            "joint_flip_gap": float(joint_gap),
            "joint_flip_gap_relative": float(joint_gap / max(energy_span, ENERGY_EQUALITY_ATOL)),
            "joint_flip_nonadditivity": float(nonadditivity),
            "joint_flip_nonadditivity_relative": float(nonadditivity / max(energy_span, ENERGY_EQUALITY_ATOL)),
            "G_ij_decomposition_error": float(abs(gij - exit_gap)),
            **{f"E_{state}": state_results[state]["energy"] for state in PAIR_STATES},
            **{f"delta_{state}": state_results[state]["energy"] - exact_energy for state in PAIR_STATES},
            **{f"count_{state}": state_results[state]["count"] for state in PAIR_STATES},
            **{f"bitstring_{state}": state_results[state]["bitstring"] for state in PAIR_STATES},
            "best_pair_state": best_state,
            "second_best_pair_state": second_state,
            "states_at_best": states_at_best,
            "G_ij": gij,
            "G_ij_relative": gij / max(energy_span, ENERGY_EQUALITY_ATOL),
            "swap_state_energy_difference": delta_01_10,
            "swap_state_difference_relative": delta_01_10 / max(energy_span, ENERGY_EQUALITY_ATOL),
            "top2_are_01_10": bool(top2_swap),
            "second_best_conditional_bitstring": state_results[second_state]["bitstring"],
            "second_best_selected_assets": state_results[second_state]["selected_assets"],
            "hamming_distance_from_global_optimum": hamming,
            "number_of_asset_swaps": int(n_swaps),
            "assets_removed": tuple(removed),
            "assets_added": tuple(added),
            "G_ij_per_swap": gij / max(n_swaps, 1),
            "old_margin_i": float(margin_lookup[i]),
            "old_margin_j": float(margin_lookup[j]),
            "old_pair_margin_min": float(min(margin_lookup[i], margin_lookup[j])),
            "old_pair_margin_sum": float(margin_lookup[i] + margin_lookup[j]),
            "opposite_decisions": bool(exact_x[i] != exact_x[j]),
        })

    table = pd.DataFrame(rows)
    table["is_degenerate"] = table["states_at_best"].map(len).gt(1)
    table["is_ambiguous"] = (~table["is_degenerate"] & table["G_ij_relative"].le(COMPETITIVE_REL_TOL))
    table["is_competitive_swap"] = (
        table["top2_are_01_10"] & table["swap_state_difference_relative"].le(COMPETITIVE_REL_TOL)
    )

    def classify(row):
        if row["is_degenerate"]:
            return "degenerate"
        if row["is_ambiguous"]:
            return "ambiguous"
        if row["is_competitive_swap"]:
            return "competitive_swap"
        return row["decision_pattern"]

    table["pair_class"] = table.apply(classify, axis=1)
    table = table.sort_values(["G_ij", "pair"], ascending=[False, True]).reset_index(drop=True)
    table["G_ij_rank"] = np.arange(1, len(table) + 1, dtype=int)
    return table


def build_classical_analysis(k_value):
    enumeration = enumerate_portfolios(k_value)
    exact_energy = float(enumeration.loc[0, "objective"])
    optimal_mask = np.isclose(
        enumeration["objective"].to_numpy(dtype=float), exact_energy,
        atol=ENERGY_EQUALITY_ATOL, rtol=0.0,
    )
    optimal = enumeration.loc[optimal_mask].copy()
    exact_asset_bitstrings = sorted(optimal["bitstring_asset_order"].astype(str).unique())
    exact_qiskit_bitstrings = sorted(optimal["bitstring_qiskit_order"].astype(str).unique())
    exact_x = np.asarray([int(v) for v in exact_asset_bitstrings[0]], dtype=int)
    energy_span = float(enumeration["objective"].max() - exact_energy)
    margins = old_asset_decision_margins(exact_x, exact_energy, energy_span)
    pairs = build_pair_gap_table(enumeration, exact_x, exact_energy, energy_span, margins)
    return {
        "k": int(k_value),
        "enumeration_df": enumeration,
        "optimal_df": optimal,
        "exact_energy": exact_energy,
        "energy_span": energy_span,
        "exact_x": exact_x,
        "exact_asset_bitstrings": exact_asset_bitstrings,
        "exact_qiskit_bitstrings": exact_qiskit_bitstrings,
        "selected_assets": selected_assets_from_x(exact_x),
        "asset_decision_df": margins,
        "pair_table": pairs,
    }


## 4. Caso original $k=4$: auditoria histórica e análise dos 45 pares

O caso $k=4$ é executado primeiro porque possui energia e bitstring históricos conhecidos. A auditoria é dividida em duas camadas independentes:

1. **Auditoria estrutural obrigatória:** contagem das 210 carteiras, 45 pares, estados condicionados, não negatividade dos gaps, paridade da distância de Hamming e decomposição de $G_{ij}$. Essa camada sempre precisa passar.
2. **Auditoria histórica opcional:** compara energia e bitstring com o experimento original. Ela só bloqueia a execução quando `STRICT_HISTORICAL_AUDIT=True`.

Com `STRICT_HISTORICAL_AUDIT=False`, o notebook pode ser executado em outras janelas temporais, matrizes de covariância e Hamiltonianos sem editar manualmente os valores históricos. As divergências históricas continuam registradas no relatório de auditoria, mas não são tratadas como falha estrutural.


In [ ]:
# ============================================================
# 4. EXECUTAR E AUDITAR O CASO ORIGINAL k=4
# ============================================================

target_classical = build_classical_analysis(TARGET_K)
enumeration_df = target_classical["enumeration_df"]
asset_decision_df = target_classical["asset_decision_df"]
pair_table = target_classical["pair_table"]
EXACT_ENERGY = target_classical["exact_energy"]
ENERGY_SPAN = target_classical["energy_span"]
EXACT_X = target_classical["exact_x"]
EXACT_ASSET_BITSTRINGS = target_classical["exact_asset_bitstrings"]
EXACT_QISKIT_BITSTRINGS = target_classical["exact_qiskit_bitstrings"]
EXACT_SELECTED_ASSETS = target_classical["selected_assets"]

energy_matches_history = np.isclose(EXACT_ENERGY, KNOWN_EXACT_ENERGY, atol=1e-10, rtol=0.0)
bitstring_matches_history = KNOWN_EXACT_QISKIT_BITSTRING in EXACT_QISKIT_BITSTRINGS
expected_counts = {
    "00": math.comb(N_ASSETS - 2, TARGET_K),
    "01": math.comb(N_ASSETS - 2, TARGET_K - 1),
    "10": math.comb(N_ASSETS - 2, TARGET_K - 1),
    "11": math.comb(N_ASSETS - 2, TARGET_K - 2),
}
energy_columns = [f"E_{state}" for state in PAIR_STATES]
delta_columns = [f"delta_{state}" for state in PAIR_STATES]

historical_audit = {
    "historical_energy_match_ok": bool(energy_matches_history),
    "historical_bitstring_match_ok": bool(bitstring_matches_history),
}

structural_audit = {
    "portfolio_count_ok": len(enumeration_df) == math.comb(N_ASSETS, TARGET_K),
    "pair_count_ok": len(pair_table) == math.comb(N_ASSETS, 2),
    "state_counts_ok": all((pair_table[f"count_{state}"] == count).all() for state, count in expected_counts.items()),
    "conditional_minimum_is_global_ok": bool(np.allclose(
        pair_table[energy_columns].min(axis=1), EXACT_ENERGY,
        atol=ENERGY_EQUALITY_ATOL, rtol=0.0,
    )),
    "nonnegative_deltas_ok": bool((pair_table[delta_columns].min(axis=1) >= -ENERGY_EQUALITY_ATOL).all()),
    "even_hamming_distance_ok": bool((pair_table["hamming_distance_from_global_optimum"] % 2 == 0).all()),
    "G_ij_decomposition_ok": bool(pair_table["G_ij_decomposition_error"].le(ENERGY_EQUALITY_ATOL).all()),
}

audit = {
    "strict_historical_audit": bool(STRICT_HISTORICAL_AUDIT),
    **historical_audit,
    **structural_audit,
}

failed_structural = [
    name
    for name, value in structural_audit.items()
    if name.endswith("_ok") and not value
]
if failed_structural:
    raise RuntimeError(
        f"Auditoria estrutural clássica falhou: {failed_structural}"
    )

failed_historical = [
    name
    for name, value in historical_audit.items()
    if name.endswith("_ok") and not value
]
if STRICT_HISTORICAL_AUDIT and failed_historical:
    raise RuntimeError(
        "Auditoria histórica clássica falhou: "
        f"{failed_historical}. Revise CSV, ordem das colunas, "
        "retorno=sum(), q e risk_free, ou use "
        "STRICT_HISTORICAL_AUDIT=False para analisar outro Hamiltoniano."
    )

if failed_historical and not STRICT_HISTORICAL_AUDIT:
    warnings.warn(
        "Os valores atuais não coincidem com o caso histórico original, "
        "mas a execução continuará porque STRICT_HISTORICAL_AUDIT=False. "
        f"Divergências: {failed_historical}",
        RuntimeWarning,
    )

print("energia exata:", EXACT_ENERGY)
print("bitstring(s) Qiskit:", EXACT_QISKIT_BITSTRINGS)
print("ativos selecionados:", EXACT_SELECTED_ASSETS)
print(json.dumps(audit, indent=2, ensure_ascii=False))
display(enumeration_df.head(10))
display(asset_decision_df)
display(pair_table.head(15))


## 5. Baseline antigo, categorias e pares de referência

A antiga `decision_margin` permanece apenas para ablação. Ela não seleciona mais sozinha os pares do experimento.

A seleção objetiva inclui:

- maior $G_{ij}$;
- maior custo de inversão conjunta;
- maior não aditividade;
- exemplos inclusão–exclusão, inclusão–inclusão e exclusão–exclusão;
- par competitivo e par ambíguo;
- AMD/COST, AIG/DAL e COST/DAL apenas como referências históricas.


In [ ]:
# ============================================================
# 5. RANKING, ABLAÇÃO E VISUALIZAÇÃO CLÁSSICA
# ============================================================


def first_or_none(df, sort_column, ascending=False):
    if df.empty:
        return None
    return df.sort_values(sort_column, ascending=ascending).iloc[0]


def selection_record(role, row):
    if row is None:
        return {"selection_role": role, "pair": None, "status": "not_found"}
    return {
        "selection_role": role,
        "pair": row["pair"],
        "asset_i": row["asset_i"],
        "asset_j": row["asset_j"],
        "asset_index_i": int(row["asset_index_i"]),
        "asset_index_j": int(row["asset_index_j"]),
        "decision_pattern": row["decision_pattern"],
        "pair_class": row["pair_class"],
        "G_ij": float(row["G_ij"]),
        "G_ij_relative": float(row["G_ij_relative"]),
        "joint_flip_gap": float(row["joint_flip_gap"]),
        "joint_flip_nonadditivity": float(row["joint_flip_nonadditivity"]),
        "old_pair_margin_min": float(row["old_pair_margin_min"]),
        "number_of_asset_swaps": int(row["number_of_asset_swaps"]),
        "status": "selected",
    }

nondegenerate = pair_table.loc[~pair_table["is_degenerate"]]
selections = [
    selection_record("largest_G_pair", first_or_none(nondegenerate, "G_ij")),
    selection_record("largest_joint_flip_gap", first_or_none(nondegenerate, "joint_flip_gap")),
    selection_record(
        "largest_abs_joint_nonadditivity",
        None if nondegenerate.empty else nondegenerate.loc[nondegenerate["joint_flip_nonadditivity"].abs().idxmax()],
    ),
    selection_record(
        "strong_exclude_include_pair",
        first_or_none(nondegenerate.loc[nondegenerate["decision_pattern"].isin(["exclude_include", "include_exclude"])], "G_ij"),
    ),
    selection_record("strong_include_include_pair", first_or_none(nondegenerate.loc[nondegenerate["decision_pattern"].eq("include_include")], "G_ij")),
    selection_record("strong_exclude_exclude_pair", first_or_none(nondegenerate.loc[nondegenerate["decision_pattern"].eq("exclude_exclude")], "G_ij")),
    selection_record("competitive_pair", first_or_none(pair_table.loc[pair_table["is_competitive_swap"]], "swap_state_difference_relative", ascending=True)),
    selection_record("most_ambiguous_pair", first_or_none(nondegenerate, "G_ij_relative", ascending=True)),
]
for role, names in {
    "reference_AMD_COST": {"AMD", "COST"},
    "reference_AIG_DAL": {"AIG", "DAL"},
    "reference_COST_DAL": {"COST", "DAL"},
}.items():
    match = pair_table.loc[pair_table.apply(lambda row: {row["asset_i"], row["asset_j"]} == names, axis=1)]
    selections.append(selection_record(role, None if match.empty else match.iloc[0]))

pair_category_selection_df = pd.DataFrame(selections)
rho_min, p_min = spearmanr(pair_table["G_ij"], pair_table["old_pair_margin_min"])
rho_sum, p_sum = spearmanr(pair_table["G_ij"], pair_table["old_pair_margin_sum"])
ablation_summary_df = pd.DataFrame([
    {"baseline": "old_pair_margin_min", "spearman_rho_with_G_ij": rho_min, "p_value": p_min},
    {"baseline": "old_pair_margin_sum", "spearman_rho_with_G_ij": rho_sum, "p_value": p_sum},
])

top_pairs = pair_table.nlargest(15, "G_ij").sort_values("G_ij")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_pairs["pair"], top_pairs["G_ij"])
ax.set_xlabel(r"$G_{ij}$")
ax.set_ylabel("par de ativos")
ax.set_title("15 maiores gaps condicionais — k=4")
fig.tight_layout()
fig.savefig(figure_dir / "01_ranking_Gij_k4.png", dpi=180, bbox_inches="tight")
plt.show()

gap_matrix = np.full((N_ASSETS, N_ASSETS), np.nan, dtype=float)
for _, row in pair_table.iterrows():
    i, j = int(row["asset_index_i"]), int(row["asset_index_j"])
    gap_matrix[i, j] = gap_matrix[j, i] = float(row["G_ij"])
np.fill_diagonal(gap_matrix, 0.0)
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(gap_matrix)
ax.set_xticks(range(N_ASSETS), tickers, rotation=45, ha="right")
ax.set_yticks(range(N_ASSETS), tickers)
ax.set_title(r"Mapa dos gaps condicionais $G_{ij}$ — k=4")
fig.colorbar(image, ax=ax, label=r"$G_{ij}$")
fig.tight_layout()
fig.savefig(figure_dir / "02_heatmap_Gij_k4.png", dpi=180, bbox_inches="tight")
plt.show()

display(pair_category_selection_df)
display(ablation_summary_df)


## Decisão científica após a parte clássica

O ranking clássico **não é interpretado como ranking de portas**. Ele apenas define pares e categorias para testes quânticos posteriores.

A ponte que ainda precisa ser testada é:

\[
G_{ij}\;\overset{?}{\longrightarrow}\;
\text{sensibilidade de }\langle H\rangle\text{ e }P(x^*)
\;\overset{?}{\longrightarrow}\;
\text{melhoria real do VQE}.
\]

A partir daqui, cada resultado deve manter juntos:

- identidade do Hamiltoniano;
- cardinalidade $k$;
- estrutura física do ansatz;
- vetor $\theta$;
- energia e probabilidade avaliadas exatamente.


# Parte II — codificação QUBO/Ising e auditoria quântica

Esta seção importa Qiskit e DOcplex somente depois de a análise clássica ter passado. O Hamiltoniano deve reproduzir exatamente a enumeração e conter apenas termos diagonais $I$, $Z$ e $ZZ$.


In [ ]:
# ============================================================
# 6. DEPENDÊNCIAS QUÂNTICAS
# ============================================================

from docplex.mp.model import Model
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.primitives import BackendEstimatorV2
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import QuadraticProgramToQubo

try:
    from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
    from qiskit_algorithms.optimizers import COBYLA
except ImportError:
    from qiskit.algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
    from qiskit.algorithms.optimizers import COBYLA

try:
    import cvxpy as cp
except ImportError:
    cp = None
    warnings.warn("CVXPY não está instalado; a enumeração continuará sendo a referência clássica.")


In [ ]:
# ============================================================
# 7. CONSTRUIR QUBO/ISING PARA UMA CARDINALIDADE
# ============================================================


def build_docplex_ising(k_value):
    model = Model(name=f"portfolio_n{N_ASSETS}_k{k_value}")
    variables = np.array([
        model.binary_var(name=f"x_{index}") for index in range(N_ASSETS)
    ], dtype=object)

    risk_expression = model.sum(
        float(sigma[row, column]) * variables[row] * variables[column]
        for row in range(N_ASSETS) for column in range(N_ASSETS)
    )
    return_expression = model.sum(
        float(mu[index]) * variables[index] for index in range(N_ASSETS)
    )
    model.minimize(
        Q_VALUE * risk_expression
        - (1.0 - Q_VALUE) * return_expression
        + RISK_FREE
    )
    model.add_constraint(model.sum(variables.tolist()) == int(k_value), ctname="budget")

    quadratic_program = from_docplex_mp(model=model)
    qubo = QuadraticProgramToQubo().convert(quadratic_program)
    ising, offset = qubo.to_ising()

    labels = [str(label) for label in ising.paulis.to_labels()]
    non_diagonal = [label for label in labels if "X" in label or "Y" in label]
    if non_diagonal:
        raise RuntimeError(f"Hamiltoniano não diagonal: {non_diagonal[:10]}")

    exact_result = NumPyMinimumEigensolver().compute_minimum_eigenvalue(operator=ising)
    exact_energy = float(np.real(exact_result.eigenvalue + offset))
    state_dict = exact_result.eigenstate.to_dict()
    exact_bitstrings = sorted({
        str(bitstring).replace(" ", "")
        for bitstring, amplitude in state_dict.items()
        if abs(amplitude) > 1e-12
    })

    return {
        "model": model,
        "quadratic_program": quadratic_program,
        "qubo": qubo,
        "ising": ising,
        "offset": float(offset),
        "pauli_labels": labels,
        "exact_energy_ising": exact_energy,
        "exact_bitstrings_ising": exact_bitstrings,
    }


target_encoding = build_docplex_ising(TARGET_K)
if not np.isclose(target_encoding["exact_energy_ising"], EXACT_ENERGY, atol=1e-10, rtol=0.0):
    raise RuntimeError("A energia Ising não coincide com a enumeração clássica.")
if not set(EXACT_QISKIT_BITSTRINGS).intersection(target_encoding["exact_bitstrings_ising"]):
    raise RuntimeError("O bitstring Ising não coincide com a enumeração clássica.")

lp_path = quantum_dir / "portfolio_k4.lp"
lp_path.write_text(target_encoding["model"].export_as_lp_string(), encoding="utf-8")

print("energia clássica:", EXACT_ENERGY)
print("energia Ising + offset:", target_encoding["exact_energy_ising"])
print("bitstrings Ising:", target_encoding["exact_bitstrings_ising"])
print("termos de Pauli:", len(target_encoding["pauli_labels"]))
print("LP salvo em:", lp_path.resolve())


# Parte III — ansatz de Dicke e mapa físico dos parâmetros

O índice `theta_index` não é tratado como propriedade física. Cada parâmetro é descrito por:

- bloco original do ansatz (`CY` ou `CCY`);
- qubits e distância do bloco;
- instruções primitivas após decomposição;
- número de ocorrências;
- período angular máximo/garantido.

Nesta implementação:

- `CY` contém uma `CRY(θ)`: período máximo $4\pi$, classe `four_pi_eligible`;
- `CCY` contém `RY(θ)`, `CCX`, `RY(-θ)`, `CCX`: equivale a `CCRY(2θ)`, possui período $2\pi$ garantido e duas ocorrências primitivas.


In [ ]:
# ============================================================
# 8. PORTAS E ANSATZ DE DICKE RASTREÁVEL
# ============================================================


def CY_parameterized(identifier):
    param = ParameterVector(name=f"x{identifier}", length=1)
    qc = QuantumCircuit(2)
    qc.cry(param[0], 1, 0)
    return qc.to_gate(label="CY")


def CCY_parameterized(identifier):
    param = ParameterVector(name=f"y{identifier}", length=1)
    qc = QuantumCircuit(3)
    qc.ry(param[0], 0)
    qc.ccx(2, 1, 0)
    qc.ry(-param[0], 0)
    qc.ccx(2, 1, 0)
    return qc.to_gate(label="CCY")


def dicke_parameter_count(n_value, k_value):
    return int(k_value * (2 * n_value - k_value - 1) / 2)


def build_tracked_dicke_ansatz(n_value, k_value, seed):
    numpy_state = np.random.get_state()
    try:
        np.random.seed(int(seed))
        qr = QuantumRegister(n_value, "q")
        qc = QuantumCircuit(qr)
        for excitation_index in range(k_value):
            qc.x(n_value - excitation_index - 1)

        records = []
        aux = 1
        for l_value in range(n_value)[::-1]:
            for i_value in range(l_value - 1, l_value - 1 - k_value, -1):
                if i_value >= 0:
                    unique_name = f"{l_value}{i_value}{aux}{np.random.randint(0, int(1e8))}"
                    if i_value == l_value - 1:
                        gate = CY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CY"
                    else:
                        gate = CCY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[i_value + 1], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CCY"
                    records.append({
                        "parameter_object": gate_parameter,
                        "parameter_name": str(gate_parameter),
                        "l": int(l_value),
                        "i": int(i_value),
                        "distance": int(l_value - i_value),
                        "ansatz_gate_type": gate_type,
                    })
                aux += 1
    finally:
        np.random.set_state(numpy_state)

    ordered_parameters = list(qc.parameters)
    parameter_to_index = {parameter: index for index, parameter in enumerate(ordered_parameters)}
    structure_rows = []
    for record in records:
        parameter_object = record.pop("parameter_object")
        structure_rows.append({
            "theta_index": int(parameter_to_index[parameter_object]),
            **record,
        })
    structure_df = pd.DataFrame(structure_rows).sort_values("theta_index").reset_index(drop=True)
    expected = dicke_parameter_count(n_value, k_value)
    if len(structure_df) != expected:
        raise RuntimeError(f"Esperados {expected} parâmetros; encontrados {len(structure_df)}.")
    structure_df["n"] = int(n_value)
    structure_df["k"] = int(k_value)
    structure_df["rho_k_over_n"] = k_value / n_value
    structure_df["u_l"] = structure_df["l"] / max(n_value - 1, 1)
    structure_df["u_distance"] = structure_df["distance"] / max(k_value, 1)
    structure_df["u_theta_index"] = structure_df["theta_index"] / max(expected - 1, 1)
    return qc.decompose(), structure_df


In [ ]:
# ============================================================
# 9. MAPA FÍSICO PÓS-DECOMPOSIÇÃO E AUDITORIA DE PERIODICIDADE
# ============================================================


def build_physical_parameter_map(ansatz, structure_df):
    parameter_order = list(ansatz.parameters)
    parameter_to_index = {parameter: index for index, parameter in enumerate(parameter_order)}
    occurrence_rows = []

    for instruction_position, instruction in enumerate(ansatz.data):
        operation = instruction.operation
        operation_name = str(operation.name).lower()
        qubits = tuple(int(ansatz.find_bit(qubit).index) for qubit in instruction.qubits)
        for parameter_slot, expression in enumerate(operation.params):
            expression_parameters = getattr(expression, "parameters", set())
            for parameter in expression_parameters:
                if parameter not in parameter_to_index:
                    continue
                try:
                    coefficient = float(expression.gradient(parameter))
                except Exception:
                    coefficient = np.nan
                occurrence_rows.append({
                    "theta_index": int(parameter_to_index[parameter]),
                    "parameter_name": str(parameter),
                    "primitive_operation": operation_name,
                    "primitive_qubits": qubits,
                    "instruction_position": int(instruction_position),
                    "parameter_slot": int(parameter_slot),
                    "parameter_coefficient": coefficient,
                })

    occurrence_df = pd.DataFrame(occurrence_rows)
    rows = []
    for theta_index, group in occurrence_df.groupby("theta_index"):
        operations = sorted(set(group["primitive_operation"].astype(str)))
        touched_qubits = sorted({
            int(qubit) for qubit_tuple in group["primitive_qubits"] for qubit in qubit_tuple
        })
        if "cry" in operations:
            primitive_type = "CRY"
            angular_period = float(4 * np.pi)
            periodicity_class = "four_pi_eligible"
        else:
            primitive_type = "RY"
            angular_period = float(2 * np.pi)
            periodicity_class = "guaranteed_2pi"
        rows.append({
            "theta_index": int(theta_index),
            "primitive_physical_type": primitive_type,
            "primitive_operations": tuple(operations),
            "physical_qubits": tuple(touched_qubits),
            "angular_period": angular_period,
            "periodicity_structural_class": periodicity_class,
            "n_occurrences_decomposed": int(len(group)),
            "parameter_coefficients": tuple(float(v) for v in group["parameter_coefficient"]),
            "first_instruction": int(group["instruction_position"].min()),
            "last_instruction": int(group["instruction_position"].max()),
            "fourier_n_grid_minimum": 5,
        })

    parameter_map = (
        structure_df.merge(pd.DataFrame(rows), on="theta_index", how="left", validate="one_to_one")
        .sort_values("theta_index")
        .reset_index(drop=True)
    )
    if parameter_map["primitive_physical_type"].isna().any():
        raise RuntimeError("Mapa físico incompleto após a decomposição.")

    cy = parameter_map["ansatz_gate_type"].eq("CY")
    ccy = parameter_map["ansatz_gate_type"].eq("CCY")
    checks = {
        "CY_is_CRY": bool(parameter_map.loc[cy, "primitive_physical_type"].eq("CRY").all()),
        "CY_has_one_occurrence": bool(parameter_map.loc[cy, "n_occurrences_decomposed"].eq(1).all()),
        "CY_has_4pi_max_period": bool(np.allclose(parameter_map.loc[cy, "angular_period"], 4 * np.pi)),
        "CCY_is_RY": bool(parameter_map.loc[ccy, "primitive_physical_type"].eq("RY").all()),
        "CCY_has_two_occurrences": bool(parameter_map.loc[ccy, "n_occurrences_decomposed"].eq(2).all()),
        "CCY_has_2pi_period": bool(np.allclose(parameter_map.loc[ccy, "angular_period"], 2 * np.pi)),
    }
    failed = [name for name, value in checks.items() if not value]
    if failed:
        raise RuntimeError(f"Auditoria estrutural falhou: {failed}")
    return parameter_map, occurrence_df, checks


target_ansatz, target_structure = build_tracked_dicke_ansatz(
    N_ASSETS, TARGET_K, RANDOM_SEED + 100 * N_ASSETS + TARGET_K
)
target_parameter_map, target_occurrences, periodicity_audit = build_physical_parameter_map(
    target_ansatz, target_structure
)

print("parâmetros k=4:", target_ansatz.num_parameters)
print(json.dumps(periodicity_audit, indent=2, ensure_ascii=False))
display(target_parameter_map)


# Parte IV — contextos multi-$k$

Agora a mesma instância financeira é reconstruída para $k=2,3,4,5$. Cada cardinalidade recebe um `problem_id` próprio e mantém juntos:

- enumeração exata e tabela $G_{ij}$;
- QUBO/Ising e offset;
- solução exata;
- ansatz de Dicke;
- mapa físico dos parâmetros;
- simulador e estimador.

Isso impede misturar $\theta$ de um circuito com o Hamiltoniano de outro problema.


In [ ]:
# ============================================================
# 10. CONSTRUIR CONTEXTOS MULTI-k
# ============================================================


def build_problem_context(k_value, classical_analysis):
    encoding = build_docplex_ising(k_value)
    if not np.isclose(
        encoding["exact_energy_ising"], classical_analysis["exact_energy"],
        atol=1e-10, rtol=0.0,
    ):
        raise RuntimeError(f"k={k_value}: Ising e enumeração não coincidem.")
    if not set(classical_analysis["exact_qiskit_bitstrings"]).intersection(
        encoding["exact_bitstrings_ising"]
    ):
        raise RuntimeError(f"k={k_value}: bitstrings Ising e clássicos não coincidem.")

    ansatz, structure_df = build_tracked_dicke_ansatz(
        N_ASSETS, k_value, RANDOM_SEED + 100 * N_ASSETS + k_value
    )
    parameter_map, occurrence_df, structure_audit = build_physical_parameter_map(ansatz, structure_df)
    simulator = AerSimulator(method="statevector", device="CPU")
    estimator_seed = int(RANDOM_SEED + 500_000 + 100 * int(k_value))
    estimator = BackendEstimatorV2(
        backend=simulator,
        options={
            "default_precision": float(ESTIMATOR_PRECISION),
            "abelian_grouping": True,
            "seed_simulator": estimator_seed,
        },
    )
    problem_id = f"{DATA_HASH}_n{N_ASSETS}_k{k_value}_q{Q_VALUE:.3f}"

    return {
        "problem_id": problem_id,
        "hamiltonian_id": DATA_HASH,
        "n": N_ASSETS,
        "k": int(k_value),
        "tickers": tickers,
        "assets_returns": assets_returns.copy(),
        "covariance": covariance.copy(),
        "classical": classical_analysis,
        "model": encoding["model"],
        "qubo": encoding["qubo"],
        "ising": encoding["ising"],
        "offset": encoding["offset"],
        "pauli_labels": encoding["pauli_labels"],
        "exact_energy": classical_analysis["exact_energy"],
        "exact_bitstrings": classical_analysis["exact_qiskit_bitstrings"],
        "ansatz": ansatz,
        "n_parameters": int(ansatz.num_parameters),
        "structure_df": structure_df,
        "parameter_map": parameter_map,
        "occurrence_df": occurrence_df,
        "structure_audit": structure_audit,
        "simulator": simulator,
        "estimator": estimator,
        "estimator_precision": float(ESTIMATOR_PRECISION),
        "estimator_shots_equivalent": int(SHOTS),
        "estimator_seed": estimator_seed,
    }


classical_by_k = {TARGET_K: target_classical}
for k_value in K_VALUES:
    if k_value not in classical_by_k:
        classical_by_k[k_value] = build_classical_analysis(k_value)

problem_contexts = {
    k_value: build_problem_context(k_value, classical_by_k[k_value])
    for k_value in K_VALUES
}

context_summary_df = pd.DataFrame([
    {
        "problem_id": context["problem_id"],
        "hamiltonian_id": context["hamiltonian_id"],
        "n": context["n"],
        "k": context["k"],
        "n_valid_portfolios": len(context["classical"]["enumeration_df"]),
        "n_parameters": context["n_parameters"],
        "exact_energy": context["exact_energy"],
        "exact_bitstrings": context["exact_bitstrings"],
        "selected_assets": context["classical"]["selected_assets"],
        "largest_G_pair": context["classical"]["pair_table"].iloc[0]["pair"],
        "largest_G": float(context["classical"]["pair_table"].iloc[0]["G_ij"]),
    }
    for context in problem_contexts.values()
]).sort_values("k")

display(context_summary_df)


## Auditoria do subespaço de Dicke

Antes de gerar bancos, o circuito é testado com um vetor aleatório por cardinalidade. Toda a probabilidade deve permanecer no subespaço de peso de Hamming $k$. Essa auditoria verifica diretamente a propriedade que justifica usar o ansatz de Dicke para seleção de portfólio com cardinalidade fixa.


In [ ]:
# ============================================================
# 10.1 AUDITORIA DE PRESERVAÇÃO DA CARDINALIDADE
# ============================================================

cardinality_audit_rows = []
for k_value, context in problem_contexts.items():
    local_rng = np.random.default_rng(RANDOM_SEED + 90_000 + k_value)
    theta_test = local_rng.uniform(-np.pi, np.pi, size=context["n_parameters"])
    assigned = context["ansatz"].assign_parameters(theta_test, inplace=False)
    probabilities = Statevector.from_instruction(assigned).probabilities_dict()
    leakage = float(sum(
        probability
        for bitstring, probability in probabilities.items()
        if str(bitstring).replace(" ", "").count("1") != k_value
    ))
    cardinality_audit_rows.append({
        "problem_id": context["problem_id"],
        "k": int(k_value),
        "probability_outside_hamming_weight_k": leakage,
        "cardinality_preserved": bool(leakage <= 1e-10),
    })

cardinality_audit_df = pd.DataFrame(cardinality_audit_rows).sort_values("k")
if not cardinality_audit_df["cardinality_preserved"].all():
    raise RuntimeError("O ansatz apresentou vazamento para fora do subespaço de Dicke.")

display(cardinality_audit_df)


# Parte V — banco definitivo de parâmetros $\theta$

Esta etapa usa apenas um desenho:

- 100 reinicializações por cardinalidade;
- 250 avaliações COBYLA por reinicialização;
- nenhum aquecimento de subconjunto;
- avaliação final exata por `Statevector`;
- amostragem de 4096 shots mantida apenas como observável adicional.

`GENERATE_OR_RESUME_BANKS=False` não cria uma versão piloto. Apenas permite executar as partes anteriores sem iniciar automaticamente 400 otimizações. Ao ativar a flag, o notebook retoma checkpoints e completa exatamente o banco definido acima.


In [ ]:
# ============================================================
# 11. MÉTRICAS EXATAS E UMA RODADA COBYLA
# ============================================================


def exact_theta_metrics(context, theta):
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if len(theta) != context["n_parameters"]:
        raise ValueError(
            f"Vetor theta com {len(theta)} elementos; esperados {context['n_parameters']}."
        )
    assigned = context["ansatz"].assign_parameters(theta, inplace=False)
    state = Statevector.from_instruction(assigned)
    energy = float(np.real(state.expectation_value(context["ising"])) + context["offset"])
    probabilities = {
        str(key).replace(" ", ""): float(value)
        for key, value in state.probabilities_dict().items()
    }
    p_opt = float(sum(probabilities.get(bitstring, 0.0) for bitstring in context["exact_bitstrings"]))
    dominant = max(probabilities, key=probabilities.get)
    return {
        "energy_exact_eval": energy,
        "gap_exact_eval": abs(energy - context["exact_energy"]),
        "p_exact_eval": p_opt,
        "dominant_bitstring_exact_eval": dominant,
        "is_exact_dominant_eval": bool(dominant in set(context["exact_bitstrings"])),
    }


def run_cobyla_bank_row(context, run_index):
    local_rng = np.random.default_rng(
        RANDOM_SEED + 1_000_000 + 10_000 * context["n"] + 100 * context["k"] + int(run_index)
    )
    theta_initial = 0.5 * np.pi * local_rng.random(context["n_parameters"])
    history = []

    def objective(theta):
        theta = np.asarray(theta, dtype=float).reshape(-1)
        result = context["estimator"].run(
            pubs=[(context["ansatz"], context["ising"], theta)],
            precision=context["estimator_precision"],
        ).result()
        energy = float(
            np.real(np.asarray(result[0].data.evs).reshape(-1)[0]) + context["offset"]
        )
        history.append((theta.copy(), energy))
        return energy

    start = perf_counter()
    result = COBYLA(maxiter=COBYLA_MAXITER).minimize(fun=objective, x0=theta_initial)
    optimizer_time = perf_counter() - start
    theta_final = np.asarray(result.x, dtype=float).reshape(-1)
    exact_metrics = exact_theta_metrics(context, theta_final)

    measured = context["ansatz"].assign_parameters(theta_final, inplace=False).copy()
    measured.measure_all()
    simulator_start = perf_counter()
    counts = (
        context["simulator"].run(
            measured,
            shots=SHOTS,
            seed_simulator=RANDOM_SEED + 2_000_000 + 10_000 * context["n"] + 100 * context["k"] + int(run_index),
        ).result().get_counts()
    )
    simulator_time = perf_counter() - simulator_start
    counts = {str(key).replace(" ", ""): int(value) for key, value in counts.items()}
    total_counts = max(sum(counts.values()), 1)
    most_frequent = max(counts, key=counts.get)
    p_opt_shots = float(sum(counts.get(bit, 0) for bit in context["exact_bitstrings"]) / total_counts)

    return {
        "problem_id": context["problem_id"],
        "hamiltonian_id": context["hamiltonian_id"],
        "n": context["n"],
        "k": context["k"],
        "run": int(run_index),
        "n_parameters": context["n_parameters"],
        "initial_point": theta_initial,
        "best_parameters": theta_final,
        "objective_function_value": float(result.fun),
        "exact_energy": context["exact_energy"],
        "nfev": int(len(history)),
        "optimizer_time": float(optimizer_time),
        "simulator_time": float(simulator_time),
        "shots": int(SHOTS),
        "estimator_precision": float(context["estimator_precision"]),
        "estimator_shots_equivalent": int(context["estimator_shots_equivalent"]),
        "probability_best_answer_shots": p_opt_shots,
        "most_frequent_bitstring": most_frequent,
        "is_exact_dominant_shots": bool(most_frequent in set(context["exact_bitstrings"])),
        "counts": counts,
        **exact_metrics,
        "status": "ok",
    }


In [ ]:
# ============================================================
# 12. GERAR, RETOMAR OU CARREGAR OS BANCOS
# ============================================================

bank_frames = {}
bank_status_rows = []

for k_value, context in problem_contexts.items():
    config_dir = bank_dir / f"n{N_ASSETS}k{k_value}"
    config_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = config_dir / "bank_rows.pkl"
    final_path = config_dir / "theta_bank.pkl"

    if checkpoint_path.exists():
        rows = pd.read_pickle(checkpoint_path)
        if not isinstance(rows, list):
            rows = []
    elif final_path.exists():
        existing_df = pd.read_pickle(final_path)
        rows = existing_df.to_dict("records")
    else:
        rows = []

    completed = {
        int(row["run"]) for row in rows
        if row.get("status") == "ok"
    }

    if GENERATE_OR_RESUME_BANKS:
        for run_index in range(BANK_RUNS_PER_CONFIGURATION):
            if run_index in completed:
                continue
            row = run_cobyla_bank_row(context, run_index)
            rows.append(row)
            pd.to_pickle(rows, checkpoint_path)
            print(
                f"k={k_value} run {run_index + 1}/{BANK_RUNS_PER_CONFIGURATION} "
                f"gap={row['gap_exact_eval']:.6e} p*={row['p_exact_eval']:.4f}"
            )

    bank_df = pd.DataFrame(rows)
    if not bank_df.empty:
        bank_df = (
            bank_df.loc[bank_df["status"].eq("ok")]
            .drop_duplicates(subset=["problem_id", "run"], keep="last")
            .sort_values("run")
            .reset_index(drop=True)
        )
        bank_df.to_pickle(final_path)
    bank_frames[k_value] = bank_df
    bank_status_rows.append({
        "problem_id": context["problem_id"],
        "k": k_value,
        "target_rows": BANK_RUNS_PER_CONFIGURATION,
        "available_rows": int(len(bank_df)),
        "complete": bool(len(bank_df) >= BANK_RUNS_PER_CONFIGURATION),
        "path": str(final_path.resolve()),
    })

bank_status_df = pd.DataFrame(bank_status_rows).sort_values("k")
display(bank_status_df)

if not GENERATE_OR_RESUME_BANKS and not bank_status_df["complete"].all():
    print(
        "Os bancos ainda não estão completos. A infraestrutura e as tabelas clássicas/quânticas "
        "continuam válidas; ative GENERATE_OR_RESUME_BANKS para completar a única amostra definida."
    )


# Parte VI — máscaras fixas e cronogramas 7 → 30 → 7

O resultado anterior mostrou que `active_7` com os outros 23 parâmetros congelados aleatoriamente é rápido, mas pouco robusto. A nova pergunta é dinâmica:

> Os sete parâmetros podem ser usados para uma descida inicial barata, os 30 para navegar até a bacia correta, e os sete novamente para o refinamento final?

## Cronograma central

Duas variantes são executadas:

\[
7\;(20) \rightarrow 30\;(até\;150) \rightarrow 7\;(até\;250),
\]

\[
7\;(30) \rightarrow 30\;(até\;150) \rightarrow 7\;(até\;250).
\]

Os números entre parênteses representam orçamento de avaliações da função objetivo. A segunda fase recebe, respectivamente, 130 ou 120 avaliações; a última recebe 100.

## Controles adicionais

- `active7_random`, `active7_zero` e `active7_anchor` testam três fundos diferentes para os 23 parâmetros congelados;
- `hybrid_30to150_7to250` remove o aquecimento inicial com os sete;
- `hybrid_7x30_30to250` remove o refinamento final com os sete;
- `full_30` permanece como referência.

Em cada restart, todos os regimes compartilham os mesmos sete valores ativos iniciais. As diferenças de fundo são introduzidas apenas quando fazem parte da hipótese testada.


In [ ]:
# ============================================================
# 13. EXPERIMENTO PAREADO: MÁSCARAS E CRONOGRAMAS 7 -> 30 -> 7
# ============================================================


def _safe_optimizer_attribute(result, name, default=np.nan):
    value = getattr(result, name, default)
    if value is None:
        return default
    try:
        if isinstance(value, (bool, np.bool_)):
            return bool(value)
        if np.isscalar(value) and np.isfinite(value):
            return value.item() if hasattr(value, "item") else value
    except TypeError:
        pass
    return value


def _threshold_column_name(threshold):
    exponent = int(round(-np.log10(float(threshold))))
    return f"nfev_to_estimator_gap_1e{exponent}"


def _first_evaluation_reaching(best_gap_history, threshold):
    best_gap_history = np.asarray(best_gap_history, dtype=float)
    reached = np.flatnonzero(best_gap_history <= float(threshold))
    return int(reached[0] + 1) if len(reached) else np.nan


def _theta_array(value, n_parameters):
    """Converte vetores salvos como array, lista ou JSON para ndarray."""
    if isinstance(value, str):
        value = json.loads(value)
    theta = np.asarray(value, dtype=float).reshape(-1)
    if len(theta) != int(n_parameters):
        raise ValueError(
            f"Vetor theta com {len(theta)} elementos; esperados {n_parameters}."
        )
    return theta


def _theta_hash(theta):
    theta = np.asarray(theta, dtype=float).reshape(-1)
    payload = np.round(theta, 12).tobytes()
    return hashlib.sha256(payload).hexdigest()[:16]


def sample_final_theta(context, theta, shots, seed_simulator):
    """Amostra o circuito final; a referência exata continua sendo o Statevector."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    measured = context["ansatz"].assign_parameters(theta, inplace=False).copy()
    measured.measure_all()
    start = perf_counter()
    raw_counts = (
        context["simulator"]
        .run(measured, shots=int(shots), seed_simulator=int(seed_simulator))
        .result()
        .get_counts()
    )
    simulator_time = perf_counter() - start
    counts = {
        str(bitstring).replace(" ", ""): int(count)
        for bitstring, count in raw_counts.items()
    }
    total = max(int(sum(counts.values())), 1)
    optimal_set = set(context["exact_bitstrings"])
    p_opt_shots = float(sum(counts.get(bit, 0) for bit in optimal_set) / total)
    dominant = max(counts, key=counts.get)
    return {
        "counts_shots": counts,
        "distribution_shots": int(total),
        "probability_best_answer_shots": p_opt_shots,
        "most_frequent_bitstring_shots": str(dominant),
        "is_exact_dominant_shots": bool(dominant in optimal_set),
        "simulator_time_shots": float(simulator_time),
        "measurement_seed": int(seed_simulator),
    }


def run_cobyla_stage(context, theta_start, free_indices, max_evaluations, stage_name):
    """
    Executa uma etapa do VQE com uma máscara fixa.

    `theta_start` é o vetor físico completo recebido da etapa anterior. Somente
    `free_indices` são entregues ao COBYLA; as demais coordenadas permanecem
    exatamente fixas durante esta etapa.
    """
    theta_start = np.asarray(theta_start, dtype=float).reshape(-1).copy()
    n_parameters = int(context["n_parameters"])
    if len(theta_start) != n_parameters:
        raise ValueError(
            f"Ponto inicial com {len(theta_start)} parâmetros; esperados {n_parameters}."
        )

    free_indices = np.asarray(sorted(set(int(v) for v in free_indices)), dtype=int)
    if len(free_indices) == 0:
        raise ValueError("Uma etapa precisa liberar ao menos um parâmetro.")
    if np.any(free_indices < 0) or np.any(free_indices >= n_parameters):
        raise ValueError("Índice livre fora do intervalo do ansatz.")

    fixed_indices = np.asarray(
        sorted(set(range(n_parameters)) - set(free_indices.tolist())), dtype=int
    )
    x0 = theta_start[free_indices].copy()
    objective_history = []

    def unpack(candidate):
        theta_full = theta_start.copy()
        theta_full[free_indices] = np.asarray(candidate, dtype=float).reshape(-1)
        return theta_full

    def objective(candidate):
        theta_full = unpack(candidate)
        result_estimator = context["estimator"].run(
            pubs=[(context["ansatz"], context["ising"], theta_full)],
            precision=context["estimator_precision"],
        ).result()
        energy = float(
            np.real(np.asarray(result_estimator[0].data.evs).reshape(-1)[0])
            + context["offset"]
        )
        objective_history.append(energy)
        return energy

    start = perf_counter()
    result = COBYLA(maxiter=int(max_evaluations)).minimize(fun=objective, x0=x0)
    optimizer_time = perf_counter() - start
    theta_final = unpack(result.x)

    if len(fixed_indices) and not np.allclose(
        theta_final[fixed_indices],
        theta_start[fixed_indices],
        atol=1e-12,
        rtol=0.0,
    ):
        raise RuntimeError(f"A etapa {stage_name} alterou um parâmetro congelado.")

    history = np.asarray(objective_history, dtype=float)
    gap_history = np.abs(history - float(context["exact_energy"]))
    exact_end = exact_theta_metrics(context, theta_final)

    return {
        "stage_name": str(stage_name),
        "theta_start": theta_start,
        "theta_final": theta_final,
        "free_indices": tuple(int(v) for v in free_indices),
        "fixed_indices": tuple(int(v) for v in fixed_indices),
        "n_free_parameters": int(len(free_indices)),
        "requested_budget": int(max_evaluations),
        "actual_nfev": int(len(history)),
        "objective_history": tuple(float(v) for v in history),
        "stage_best_estimator_gap": float(np.min(gap_history)) if len(gap_history) else np.nan,
        "objective_function_value": float(result.fun),
        "optimizer_time": float(optimizer_time),
        "optimizer_nfev_reported": _safe_optimizer_attribute(result, "nfev", np.nan),
        "optimizer_iterations_reported": _safe_optimizer_attribute(result, "nit", np.nan),
        "optimizer_success_reported": _safe_optimizer_attribute(result, "success", np.nan),
        "optimizer_message": str(_safe_optimizer_attribute(result, "message", "")),
        **{f"end_{key}": value for key, value in exact_end.items()},
    }


def build_regime_specs(total_budget=STAGED_TOTAL_BUDGET):
    """Define os regimes antes de executar qualquer restart."""
    if STAGED_FULL_PHASE_END >= total_budget:
        raise ValueError("STAGED_FULL_PHASE_END deve ser menor que o orçamento total.")

    warm20, warm30 = (int(v) for v in STAGED_ACTIVE_WARMUP_BUDGETS)
    return {
        "full_30": {
            "label": "30 durante toda a busca",
            "initial_background": "random",
            # O segundo valor é o alvo acumulado de nfev, não o tamanho isolado da etapa.
            "stages": [("full_30", int(total_budget))],
        },
        "active7_random": {
            "label": "7 ativos; fundo aleatório",
            "initial_background": "random",
            "stages": [("active_7", int(total_budget))],
        },
        "active7_zero": {
            "label": "7 ativos; fundo zero",
            "initial_background": "zero_inactive",
            "stages": [("active_7", int(total_budget))],
        },
        "active7_anchor": {
            "label": "7 ativos; fundo da âncora",
            "initial_background": "anchor_inactive",
            "stages": [("active_7", int(total_budget))],
        },
        "hybrid_7x20_30to150_7to250": {
            "label": "7(20) -> 30(até 150) -> 7(até 250)",
            "initial_background": "random",
            "stages": [
                ("active_7", warm20),
                ("full_30", int(STAGED_FULL_PHASE_END)),
                ("active_7", int(total_budget)),
            ],
        },
        "hybrid_7x30_30to150_7to250": {
            "label": "7(30) -> 30(até 150) -> 7(até 250)",
            "initial_background": "random",
            "stages": [
                ("active_7", warm30),
                ("full_30", int(STAGED_FULL_PHASE_END)),
                ("active_7", int(total_budget)),
            ],
        },
        "hybrid_30to150_7to250": {
            "label": "30(até 150) -> 7(até 250)",
            "initial_background": "random",
            "stages": [
                ("full_30", int(STAGED_FULL_PHASE_END)),
                ("active_7", int(total_budget)),
            ],
        },
        "hybrid_7x30_30to250": {
            "label": "7(30) -> 30(até 250)",
            "initial_background": "random",
            "stages": [
                ("active_7", warm30),
                ("full_30", int(total_budget)),
            ],
        },
    }


def make_regime_initial_theta(common_initial, background, active_indices, anchor_theta):
    theta = np.asarray(common_initial, dtype=float).reshape(-1).copy()
    n_parameters = len(theta)
    active_indices = np.asarray(active_indices, dtype=int)
    inactive_indices = np.asarray(
        sorted(set(range(n_parameters)) - set(active_indices.tolist())), dtype=int
    )

    if background == "random":
        return theta
    if background == "zero_inactive":
        theta[inactive_indices] = 0.0
        return theta
    if background == "anchor_inactive":
        if anchor_theta is None:
            raise RuntimeError("O regime active7_anchor exige uma âncora válida.")
        anchor_theta = _theta_array(anchor_theta, n_parameters)
        theta[inactive_indices] = anchor_theta[inactive_indices]
        return theta
    raise ValueError(f"Fundo desconhecido: {background}")


def resolve_anchor_theta(context):
    """
    Obtém uma única âncora independente para fixar os 23 parâmetros.

    Prioridade:
    1. cache específico desta versão;
    2. banco multi-k já disponível;
    3. resultados `full_30` da versão anterior;
    4. calibração independente com restarts adicionais.
    """
    n_parameters = int(context["n_parameters"])
    cache_path = staged_dir / "anchor_reference.pkl"

    if cache_path.exists():
        cached = pd.read_pickle(cache_path)
        if isinstance(cached, dict) and "theta" in cached:
            theta = _theta_array(cached["theta"], n_parameters)
            metadata = dict(cached.get("metadata", {}))
            metadata.update({"source": "staged_anchor_cache", "path": str(cache_path.resolve())})
            return theta, metadata

    candidates = []

    bank_df = bank_frames.get(TARGET_K, pd.DataFrame())
    if isinstance(bank_df, pd.DataFrame) and not bank_df.empty:
        for _, row in bank_df.iterrows():
            try:
                theta = _theta_array(row["best_parameters"], n_parameters)
                gap = float(row.get("gap_exact_eval", np.inf))
                candidates.append((gap, theta, "theta_bank_k4"))
            except Exception:
                continue

    legacy_path = legacy_active7_dir / "active7_vs_full30_execution_bank.pkl"
    if legacy_path.exists():
        try:
            legacy_df = pd.read_pickle(legacy_path)
            if isinstance(legacy_df, pd.DataFrame) and not legacy_df.empty:
                legacy_df = legacy_df.loc[legacy_df.get("regime", "").eq("full_30")]
                for _, row in legacy_df.iterrows():
                    try:
                        theta = _theta_array(row["best_parameters"], n_parameters)
                        gap = float(row.get("gap_exact_eval", np.inf))
                        candidates.append((gap, theta, "legacy_full30_bank"))
                    except Exception:
                        continue
        except Exception as exc:
            warnings.warn(f"Banco legado de âncora não pôde ser lido: {exc}")

    if candidates:
        candidates.sort(key=lambda item: item[0])
        gap, theta, source = candidates[0]
        metadata = {
            "source": source,
            "gap_exact_eval": float(gap),
            "theta_hash": _theta_hash(theta),
            "calibration_restarts": 0,
        }
        pd.to_pickle({"theta": theta, "metadata": metadata}, cache_path)
        return theta, metadata

    if not RUN_STAGED_MASK_EXPERIMENT:
        return None, {
            "source": "unavailable_with_experiment_disabled",
            "theta_hash": None,
            "calibration_restarts": 0,
        }

    calibration_rows = []
    all_indices = np.arange(n_parameters, dtype=int)
    for calibration_restart in range(int(ANCHOR_CALIBRATION_RESTARTS)):
        rng = np.random.default_rng(
            RANDOM_SEED + 7_000_000 + 10_000 * TARGET_K + calibration_restart
        )
        theta_initial = 0.5 * np.pi * rng.random(n_parameters)
        stage = run_cobyla_stage(
            context=context,
            theta_start=theta_initial,
            free_indices=all_indices,
            max_evaluations=ANCHOR_CALIBRATION_MAXITER,
            stage_name=f"anchor_calibration_{calibration_restart}",
        )
        calibration_rows.append({
            "restart": int(calibration_restart),
            "theta": stage["theta_final"],
            "gap_exact_eval": float(stage["end_gap_exact_eval"]),
            "p_exact_eval": float(stage["end_p_exact_eval"]),
            "nfev": int(stage["actual_nfev"]),
        })
        print(
            f"âncora {calibration_restart + 1}/{ANCHOR_CALIBRATION_RESTARTS}: "
            f"gap={stage['end_gap_exact_eval']:.3e} "
            f"p*={stage['end_p_exact_eval']:.4f}"
        )

    calibration_df = pd.DataFrame(calibration_rows).sort_values("gap_exact_eval")
    best = calibration_df.iloc[0]
    theta = _theta_array(best["theta"], n_parameters)
    metadata = {
        "source": "independent_full30_calibration",
        "gap_exact_eval": float(best["gap_exact_eval"]),
        "p_exact_eval": float(best["p_exact_eval"]),
        "theta_hash": _theta_hash(theta),
        "calibration_restarts": int(ANCHOR_CALIBRATION_RESTARTS),
    }
    pd.to_pickle({"theta": theta, "metadata": metadata}, cache_path)
    calibration_df.to_pickle(staged_dir / "anchor_calibration_rows.pkl")
    return theta, metadata


def run_staged_regime(
    context,
    common_initial,
    anchor_theta,
    restart,
    regime,
    regime_spec,
    execution_order,
):
    n_parameters = int(context["n_parameters"])
    active_indices = np.asarray(ACTIVE_THETA_INDICES, dtype=int)
    all_indices = np.arange(n_parameters, dtype=int)
    inactive_indices = np.asarray(
        sorted(set(all_indices.tolist()) - set(active_indices.tolist())), dtype=int
    )

    theta_initial = make_regime_initial_theta(
        common_initial=common_initial,
        background=regime_spec["initial_background"],
        active_indices=active_indices,
        anchor_theta=anchor_theta,
    )
    theta_current = theta_initial.copy()
    initial_exact = exact_theta_metrics(context, theta_current)

    objective_history = []
    stage_records = []
    total_optimizer_time = 0.0
    cumulative_actual = 0
    previous_exact = initial_exact

    for stage_index, (stage_mode, cumulative_target) in enumerate(
        regime_spec["stages"], start=1
    ):
        # O orçamento desta etapa completa o total efetivamente usado até o alvo
        # acumulado. Assim, se uma etapa anterior encerrar um pouco antes, a fase
        # seguinte recebe o saldo correspondente.
        stage_budget = max(int(cumulative_target) - int(cumulative_actual), 1)
        free_indices = all_indices if stage_mode == "full_30" else active_indices
        stage = run_cobyla_stage(
            context=context,
            theta_start=theta_current,
            free_indices=free_indices,
            max_evaluations=int(stage_budget),
            stage_name=f"{regime}_stage{stage_index}_{stage_mode}",
        )

        cumulative_actual += int(stage["actual_nfev"])
        total_optimizer_time += float(stage["optimizer_time"])
        objective_history.extend(stage["objective_history"])

        record = {
            "stage_index": int(stage_index),
            "stage_mode": str(stage_mode),
            "requested_budget": int(stage_budget),
            "cumulative_requested_budget": int(cumulative_target),
            "actual_nfev": int(stage["actual_nfev"]),
            "cumulative_actual_nfev": int(cumulative_actual),
            "n_free_parameters": int(stage["n_free_parameters"]),
            "optimizer_time": float(stage["optimizer_time"]),
            "p_exact_start": float(previous_exact["p_exact_eval"]),
            "gap_exact_start": float(previous_exact["gap_exact_eval"]),
            "p_exact_end": float(stage["end_p_exact_eval"]),
            "gap_exact_end": float(stage["end_gap_exact_eval"]),
            "stage_best_estimator_gap": float(stage["stage_best_estimator_gap"]),
            "optimizer_success_reported": stage["optimizer_success_reported"],
            "optimizer_message": stage["optimizer_message"],
            "theta_start": stage["theta_start"],
            "theta_final": stage["theta_final"],
        }
        stage_records.append(record)
        theta_current = np.asarray(stage["theta_final"], dtype=float).copy()
        previous_exact = {
            key.replace("end_", ""): value
            for key, value in stage.items()
            if key.startswith("end_")
        }

    final_exact = exact_theta_metrics(context, theta_current)
    seed_payload = (
        f"{RANDOM_SEED}|{context['problem_id']}|{restart}|{regime}|"
        f"{STAGED_DISTRIBUTION_SHOTS}|{_theta_hash(theta_current)}"
    )
    measurement_seed = int(
        hashlib.sha256(seed_payload.encode("utf-8")).hexdigest()[:8], 16
    ) % (2**31 - 1)
    sampled_metrics = sample_final_theta(
        context=context,
        theta=theta_current,
        shots=STAGED_DISTRIBUTION_SHOTS,
        seed_simulator=measurement_seed,
    )

    objective_history = np.asarray(objective_history, dtype=float)
    estimator_gap_history = np.abs(objective_history - float(context["exact_energy"]))
    best_estimator_gap_history = np.minimum.accumulate(estimator_gap_history)

    threshold_metrics = {
        _threshold_column_name(threshold): _first_evaluation_reaching(
            best_estimator_gap_history, threshold
        )
        for threshold in STAGED_ESTIMATOR_GAP_THRESHOLDS
    }

    p_success = bool(final_exact["p_exact_eval"] >= STAGED_SUCCESS_P_THRESHOLD)
    gap_success = bool(final_exact["gap_exact_eval"] <= STAGED_SUCCESS_GAP_ATOL)
    strict_gap_success = bool(final_exact["gap_exact_eval"] <= STAGED_STRICT_GAP_ATOL)

    inactive_drift = theta_current[inactive_indices] - theta_initial[inactive_indices]
    active_drift = theta_current[active_indices] - theta_initial[active_indices]

    return {
        "problem_id": context["problem_id"],
        "hamiltonian_id": context["hamiltonian_id"],
        "n": int(context["n"]),
        "k": int(context["k"]),
        "restart": int(restart),
        "regime": str(regime),
        "regime_label": str(regime_spec["label"]),
        "execution_order": int(execution_order),
        "initial_background": str(regime_spec["initial_background"]),
        "schedule": tuple(
            (str(mode), int(cumulative_target))
            for mode, cumulative_target in regime_spec["stages"]
        ),
        "n_stages": int(len(regime_spec["stages"])),
        "n_parameters_total": int(n_parameters),
        "active_theta_indices": tuple(int(v) for v in active_indices),
        "inactive_theta_indices": tuple(int(v) for v in inactive_indices),
        "initial_theta": theta_initial,
        "common_initial_theta": np.asarray(common_initial, dtype=float).copy(),
        "best_parameters": theta_current,
        "stage_records": tuple(stage_records),
        "stage_actual_nfev": tuple(int(record["actual_nfev"]) for record in stage_records),
        "stage_cumulative_actual_nfev": tuple(
            int(record["cumulative_actual_nfev"]) for record in stage_records
        ),
        "stage_requested_budgets": tuple(
            int(record["requested_budget"]) for record in stage_records
        ),
        "stage_cumulative_requested_budgets": tuple(
            int(record["cumulative_requested_budget"]) for record in stage_records
        ),
        "objective_history": tuple(float(v) for v in objective_history),
        "best_estimator_gap_history": tuple(float(v) for v in best_estimator_gap_history),
        "best_estimator_gap": float(np.min(best_estimator_gap_history)),
        "objective_function_value": float(np.min(objective_history)),
        "requested_total_budget": int(regime_spec["stages"][-1][1]),
        "nfev": int(len(objective_history)),
        "interaction_count": int(len(objective_history)),
        "optimizer_time": float(total_optimizer_time),
        "initial_p_exact_eval": float(initial_exact["p_exact_eval"]),
        "initial_gap_exact_eval": float(initial_exact["gap_exact_eval"]),
        "active_parameter_step_l2": float(np.linalg.norm(active_drift)),
        "inactive_parameter_step_l2": float(np.linalg.norm(inactive_drift)),
        "inactive_parameter_drift_max": float(np.max(np.abs(inactive_drift)))
        if len(inactive_drift)
        else 0.0,
        "estimator_precision": float(context["estimator_precision"]),
        "estimator_shots_equivalent": int(context["estimator_shots_equivalent"]),
        "success_probability": p_success,
        "success_energy": gap_success,
        "success_energy_strict": strict_gap_success,
        "success_joint": bool(p_success and gap_success),
        "status": "ok",
        **threshold_metrics,
        **final_exact,
        **sampled_metrics,
    }


def pad_best_gap_history(history, target_length):
    history = np.asarray(history, dtype=float)
    if len(history) == 0:
        return np.full(int(target_length), np.nan, dtype=float)
    if len(history) >= target_length:
        return history[:target_length]
    return np.pad(history, (0, target_length - len(history)), mode="edge")


REGIME_SPECS = build_regime_specs()

staged_mask_runs_df = pd.DataFrame()
staged_mask_summary_df = pd.DataFrame()
staged_mask_stage_df = pd.DataFrame()
staged_mask_stage_summary_df = pd.DataFrame()
staged_mask_paired_wide_df = pd.DataFrame()
staged_mask_tests_df = pd.DataFrame()
staged_mask_convergence_df = pd.DataFrame()
staged_mask_metadata = {
    "enabled": bool(RUN_STAGED_MASK_EXPERIMENT),
    "executed": False,
    "active_theta_indices": [int(v) for v in ACTIVE_THETA_INDICES],
    "regime_specs": REGIME_SPECS,
}

if RUN_STAGED_MASK_EXPERIMENT:
    context = problem_contexts[TARGET_K]
    n_parameters = int(context["n_parameters"])
    active_indices = np.asarray(ACTIVE_THETA_INDICES, dtype=int)

    if len(set(active_indices.tolist())) != len(active_indices):
        raise RuntimeError("ACTIVE_THETA_INDICES contém índices repetidos.")
    if np.any(active_indices < 0) or np.any(active_indices >= n_parameters):
        raise RuntimeError(
            f"Índices ativos incompatíveis com o ansatz de {n_parameters} parâmetros."
        )
    if n_parameters != 30:
        warnings.warn(
            f"O caso k={TARGET_K} possui {n_parameters} parâmetros, não 30. "
            "Os nomes dos regimes são mantidos por compatibilidade conceitual."
        )

    anchor_theta, anchor_metadata = resolve_anchor_theta(context)
    if anchor_theta is None:
        raise RuntimeError(
            "A âncora não foi encontrada. Ative o experimento para permitir a calibração."
        )
    print("Âncora:", json.dumps(anchor_metadata, indent=2, ensure_ascii=False))

    signature_payload = {
        "problem_id": context["problem_id"],
        "active_theta_indices": [int(v) for v in active_indices],
        "restarts": int(STAGED_EXPERIMENT_RESTARTS),
        "regime_specs": REGIME_SPECS,
        "estimator_precision": float(context["estimator_precision"]),
        "distribution_shots": int(STAGED_DISTRIBUTION_SHOTS),
        "anchor_theta_hash": _theta_hash(anchor_theta),
        "schema_version": "staged_masks_7_30_7_v1",
    }
    experiment_signature = hashlib.sha256(
        json.dumps(signature_payload, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]

    checkpoint_path = staged_dir / f"staged_mask_rows_{experiment_signature}.pkl"
    if checkpoint_path.exists():
        loaded_rows = pd.read_pickle(checkpoint_path)
        rows = loaded_rows if isinstance(loaded_rows, list) else []
    else:
        rows = []

    completed = {
        (str(row["regime"]), int(row["restart"]))
        for row in rows
        if row.get("status") == "ok"
    }

    regime_names = list(REGIME_SPECS)
    for restart in range(int(STAGED_EXPERIMENT_RESTARTS)):
        rng = np.random.default_rng(
            RANDOM_SEED + 9_000_000 + 1000 * TARGET_K + restart
        )
        common_initial = 0.5 * np.pi * rng.random(n_parameters)

        # A ordem é embaralhada de forma reprodutível para reduzir viés temporal.
        order_rng = np.random.default_rng(
            RANDOM_SEED + 9_500_000 + 1000 * TARGET_K + restart
        )
        regime_order = list(order_rng.permutation(regime_names))

        for execution_order, regime in enumerate(regime_order):
            key = (regime, restart)
            if key in completed:
                continue
            row = run_staged_regime(
                context=context,
                common_initial=common_initial,
                anchor_theta=anchor_theta,
                restart=restart,
                regime=regime,
                regime_spec=REGIME_SPECS[regime],
                execution_order=execution_order,
            )
            rows.append(row)
            pd.to_pickle(rows, checkpoint_path)
            completed.add(key)
            stage_text = "+".join(str(v) for v in row["stage_actual_nfev"])
            print(
                f"restart={restart + 1:03d}/{STAGED_EXPERIMENT_RESTARTS} "
                f"regime={regime:34s} nfev={row['nfev']:3d} "
                f"stages={stage_text:11s} p*={row['p_exact_eval']:.4f} "
                f"gap={row['gap_exact_eval']:.3e}"
            )

    staged_mask_runs_df = (
        pd.DataFrame(rows)
        .loc[lambda df: df["status"].eq("ok")]
        .drop_duplicates(subset=["regime", "restart"], keep="last")
        .sort_values(["restart", "regime"])
        .reset_index(drop=True)
    )

    expected_rows = len(REGIME_SPECS) * int(STAGED_EXPERIMENT_RESTARTS)
    if len(staged_mask_runs_df) != expected_rows:
        warnings.warn(
            f"Esperadas {expected_rows} linhas; encontradas {len(staged_mask_runs_df)}."
        )

    staged_mask_summary_df = (
        staged_mask_runs_df.groupby(["regime", "regime_label"], as_index=False)
        .agg(
            n_restarts=("restart", "nunique"),
            n_stages=("n_stages", "first"),
            requested_total_budget=("requested_total_budget", "first"),
            p_exact_median=("p_exact_eval", "median"),
            p_exact_q25=("p_exact_eval", lambda s: float(s.quantile(0.25))),
            p_exact_q75=("p_exact_eval", lambda s: float(s.quantile(0.75))),
            p_shots_median=("probability_best_answer_shots", "median"),
            gap_exact_median=("gap_exact_eval", "median"),
            gap_exact_q25=("gap_exact_eval", lambda s: float(s.quantile(0.25))),
            gap_exact_q75=("gap_exact_eval", lambda s: float(s.quantile(0.75))),
            best_estimator_gap_median=("best_estimator_gap", "median"),
            nfev_median=("nfev", "median"),
            nfev_mean=("nfev", "mean"),
            optimizer_time_median=("optimizer_time", "median"),
            optimizer_time_mean=("optimizer_time", "mean"),
            exact_dominant_rate=("is_exact_dominant_eval", "mean"),
            shots_dominant_rate=("is_exact_dominant_shots", "mean"),
            success_probability_rate=("success_probability", "mean"),
            success_energy_rate=("success_energy", "mean"),
            success_energy_strict_rate=("success_energy_strict", "mean"),
            success_joint_rate=("success_joint", "mean"),
        )
    )

    for threshold in STAGED_ESTIMATOR_GAP_THRESHOLDS:
        column = _threshold_column_name(threshold)
        threshold_summary = (
            staged_mask_runs_df.groupby("regime")[column]
            .agg(
                reach_rate=lambda s: float(s.notna().mean()),
                median_when_reached=lambda s: float(s.dropna().median())
                if s.notna().any()
                else np.nan,
            )
            .reset_index()
            .rename(
                columns={
                    "reach_rate": f"{column}_reach_rate",
                    "median_when_reached": f"{column}_median_when_reached",
                }
            )
        )
        staged_mask_summary_df = staged_mask_summary_df.merge(
            threshold_summary, on="regime", how="left"
        )

    stage_rows = []
    for _, run_row in staged_mask_runs_df.iterrows():
        for record in run_row["stage_records"]:
            stage_rows.append({
                "restart": int(run_row["restart"]),
                "regime": str(run_row["regime"]),
                "regime_label": str(run_row["regime_label"]),
                **record,
            })
    staged_mask_stage_df = pd.DataFrame(stage_rows)
    staged_mask_stage_summary_df = (
        staged_mask_stage_df.groupby(
            ["regime", "regime_label", "stage_index", "stage_mode"], as_index=False
        )
        .agg(
            requested_budget=("requested_budget", "first"),
            actual_nfev_median=("actual_nfev", "median"),
            cumulative_actual_nfev_median=("cumulative_actual_nfev", "median"),
            p_exact_start_median=("p_exact_start", "median"),
            p_exact_end_median=("p_exact_end", "median"),
            gap_exact_start_median=("gap_exact_start", "median"),
            gap_exact_end_median=("gap_exact_end", "median"),
            stage_best_estimator_gap_median=("stage_best_estimator_gap", "median"),
            optimizer_time_median=("optimizer_time", "median"),
        )
    )

    metrics_for_wide = [
        "p_exact_eval",
        "probability_best_answer_shots",
        "gap_exact_eval",
        "best_estimator_gap",
        "nfev",
        "optimizer_time",
        "success_joint",
    ]
    staged_mask_paired_wide_df = staged_mask_runs_df.pivot(
        index="restart", columns="regime", values=metrics_for_wide
    )
    staged_mask_paired_wide_df.columns = [
        f"{metric}__{regime}" for metric, regime in staged_mask_paired_wide_df.columns
    ]
    staged_mask_paired_wide_df = staged_mask_paired_wide_df.reset_index()

    def paired_wilcoxon_between(regime_a, regime_b, metric, higher_is_better):
        left = staged_mask_runs_df.loc[
            staged_mask_runs_df["regime"].eq(regime_a), ["restart", metric]
        ].rename(columns={metric: "value_a"})
        right = staged_mask_runs_df.loc[
            staged_mask_runs_df["regime"].eq(regime_b), ["restart", metric]
        ].rename(columns={metric: "value_b"})
        paired = left.merge(right, on="restart", how="inner")
        a = paired["value_a"].to_numpy(dtype=float)
        b = paired["value_b"].to_numpy(dtype=float)
        finite = np.isfinite(a) & np.isfinite(b)
        a = a[finite]
        b = b[finite]
        difference = a - b
        try:
            statistic, p_value = wilcoxon(
                a,
                b,
                alternative="two-sided",
                zero_method="wilcox",
            )
        except ValueError:
            statistic, p_value = np.nan, 1.0
        if higher_is_better:
            better_fraction = float(np.mean(difference > 0.0)) if len(difference) else np.nan
        else:
            better_fraction = float(np.mean(difference < 0.0)) if len(difference) else np.nan
        return {
            "regime_a": regime_a,
            "regime_b": regime_b,
            "metric": metric,
            "higher_is_better": bool(higher_is_better),
            "n_pairs": int(len(difference)),
            "median_a": float(np.median(a)) if len(a) else np.nan,
            "median_b": float(np.median(b)) if len(b) else np.nan,
            "median_difference_a_minus_b": float(np.median(difference))
            if len(difference)
            else np.nan,
            "regime_a_better_fraction": better_fraction,
            "wilcoxon_statistic": float(statistic) if np.isfinite(statistic) else np.nan,
            "wilcoxon_p_value": float(p_value),
        }

    comparison_pairs = [
        (regime, "full_30")
        for regime in REGIME_SPECS
        if regime != "full_30"
    ] + [
        ("hybrid_7x20_30to150_7to250", "hybrid_7x30_30to150_7to250"),
        ("hybrid_7x30_30to150_7to250", "hybrid_30to150_7to250"),
        ("hybrid_7x30_30to150_7to250", "hybrid_7x30_30to250"),
    ]
    test_specs = [
        ("p_exact_eval", True),
        ("probability_best_answer_shots", True),
        ("gap_exact_eval", False),
        ("best_estimator_gap", False),
        ("nfev", False),
        ("optimizer_time", False),
    ]
    staged_mask_tests_df = pd.DataFrame([
        paired_wilcoxon_between(regime_a, regime_b, metric, higher)
        for regime_a, regime_b in comparison_pairs
        for metric, higher in test_specs
    ])

    convergence_rows = []
    for regime, group in staged_mask_runs_df.groupby("regime"):
        matrix = np.stack(
            [
                pad_best_gap_history(history, STAGED_TOTAL_BUDGET)
                for history in group["best_estimator_gap_history"]
            ],
            axis=0,
        )
        for evaluation in range(int(STAGED_TOTAL_BUDGET)):
            values = matrix[:, evaluation]
            convergence_rows.append({
                "regime": str(regime),
                "evaluation": int(evaluation + 1),
                "best_estimator_gap_median": float(np.nanmedian(values)),
                "best_estimator_gap_q25": float(np.nanquantile(values, 0.25)),
                "best_estimator_gap_q75": float(np.nanquantile(values, 0.75)),
                "best_estimator_gap_mean": float(np.nanmean(values)),
            })
    staged_mask_convergence_df = pd.DataFrame(convergence_rows)

    staged_mask_metadata = {
        "enabled": True,
        "executed": True,
        "experiment_signature": experiment_signature,
        **signature_payload,
        "anchor_metadata": anchor_metadata,
        "n_rows": int(len(staged_mask_runs_df)),
        "n_complete_restarts": int(staged_mask_runs_df["restart"].nunique()),
        "success_definition": {
            "p_exact_min": float(STAGED_SUCCESS_P_THRESHOLD),
            "gap_exact_max": float(STAGED_SUCCESS_GAP_ATOL),
            "strict_gap_exact_max": float(STAGED_STRICT_GAP_ATOL),
        },
        "primary_cost_metric": "nfev/objective evaluations",
    }

    display(staged_mask_summary_df)
    display(staged_mask_stage_summary_df)
    display(staged_mask_tests_df.head(30))

    # ---------- Curvas: fundos fixos ----------
    background_regimes = [
        "full_30",
        "active7_random",
        "active7_zero",
        "active7_anchor",
    ]
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for regime in background_regimes:
        subset = staged_mask_convergence_df.loc[
            staged_mask_convergence_df["regime"].eq(regime)
        ]
        ax.plot(
            subset["evaluation"],
            np.maximum(subset["best_estimator_gap_median"], 1e-12),
            label=REGIME_SPECS[regime]["label"],
        )
    for threshold in STAGED_ESTIMATOR_GAP_THRESHOLDS:
        ax.axhline(float(threshold), linestyle="--", linewidth=1)
    ax.set_yscale("log")
    ax.set_xlabel("avaliação da função objetivo")
    ax.set_ylabel("melhor gap estimado acumulado")
    ax.set_title("Fundos dos 23 parâmetros: aleatório, zero e âncora")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(figure_dir / "staged_masks_background_convergence.png", dpi=180)
    plt.show()

    # ---------- Curvas: cronogramas ----------
    schedule_regimes = [
        "full_30",
        "hybrid_7x20_30to150_7to250",
        "hybrid_7x30_30to150_7to250",
        "hybrid_30to150_7to250",
        "hybrid_7x30_30to250",
    ]
    fig, ax = plt.subplots(figsize=(12, 6))
    for regime in schedule_regimes:
        subset = staged_mask_convergence_df.loc[
            staged_mask_convergence_df["regime"].eq(regime)
        ]
        ax.plot(
            subset["evaluation"],
            np.maximum(subset["best_estimator_gap_median"], 1e-12),
            label=REGIME_SPECS[regime]["label"],
        )
    for boundary in sorted(set([20, 30, STAGED_FULL_PHASE_END])):
        ax.axvline(boundary, linestyle=":", linewidth=1)
    for threshold in STAGED_ESTIMATOR_GAP_THRESHOLDS:
        ax.axhline(float(threshold), linestyle="--", linewidth=1)
    ax.set_yscale("log")
    ax.set_xlabel("avaliação da função objetivo")
    ax.set_ylabel("melhor gap estimado acumulado")
    ax.set_title("Cronogramas adaptativos: 7 -> 30 -> 7")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(figure_dir / "staged_masks_schedule_convergence.png", dpi=180)
    plt.show()

    # ---------- Probabilidade exata final ----------
    ordered_regimes = list(REGIME_SPECS)
    probability_data = [
        staged_mask_runs_df.loc[
            staged_mask_runs_df["regime"].eq(regime), "p_exact_eval"
        ].to_numpy(dtype=float)
        for regime in ordered_regimes
    ]
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.boxplot(probability_data, labels=ordered_regimes, showfliers=False)
    ax.axhline(STAGED_SUCCESS_P_THRESHOLD, linestyle="--", linewidth=1)
    ax.set_ylabel("probabilidade exata da solução P(x*)")
    ax.set_title("Qualidade final por máscara e cronograma")
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    fig.savefig(figure_dir / "staged_masks_probability.png", dpi=180)
    plt.show()

    # ---------- Gap energético exato final ----------
    gap_data = [
        staged_mask_runs_df.loc[
            staged_mask_runs_df["regime"].eq(regime), "gap_exact_eval"
        ].to_numpy(dtype=float)
        for regime in ordered_regimes
    ]
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.boxplot(gap_data, labels=ordered_regimes, showfliers=False)
    ax.axhline(STAGED_SUCCESS_GAP_ATOL, linestyle="--", linewidth=1)
    ax.set_yscale("log")
    ax.set_ylabel("|E - E_exato|")
    ax.set_title("Gap energético final por máscara e cronograma")
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    fig.savefig(figure_dir / "staged_masks_exact_gap.png", dpi=180)
    plt.show()

    # ---------- Fronteira custo-qualidade ----------
    fig, ax = plt.subplots(figsize=(9, 6))
    for _, row in staged_mask_summary_df.iterrows():
        ax.scatter(row["nfev_median"], row["p_exact_median"], s=70)
        ax.annotate(
            row["regime"],
            (row["nfev_median"], row["p_exact_median"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )
    ax.set_xlabel("nfev mediano")
    ax.set_ylabel("P(x*) exata mediana")
    ax.set_title("Fronteira custo-qualidade dos regimes")
    fig.tight_layout()
    fig.savefig(figure_dir / "staged_masks_cost_quality.png", dpi=180)
    plt.show()


# Parte VII — tabelas para análise quântica e Transformer

O objetivo desta seção é separar entidades que não devem ser confundidas:

1. **problema/Hamiltoniano:** retornos, covariância, $k$, Ising, solução exata e gaps de pares;
2. **parâmetro estrutural:** porta, qubits, distância, ocorrências e período;
3. **execução VQE:** ponto inicial, $\theta$ final, energia, $P(x^*)$ e custo;
4. **ligação par–$\theta$:** quais parâmetros tocam um ou os dois qubits de cada par.

O Transformer deverá receber grupos completos por `problem_id`. Um split aleatório entre vetores do mesmo Hamiltoniano produziria vazamento.


In [ ]:
# ============================================================
# 14. CONSTRUIR BANCOS NORMALIZADOS
# ============================================================

problem_rows = []
pair_frames = []
parameter_frames = []
theta_frames = []
link_frames = []

for k_value, context in problem_contexts.items():
    ising = context["ising"]
    coeffs = [float(np.real(value)) for value in np.asarray(ising.coeffs)]
    labels = [str(value) for value in ising.paulis.to_labels()]
    classical = context["classical"]

    problem_rows.append({
        "problem_id": context["problem_id"],
        "hamiltonian_id": context["hamiltonian_id"],
        "n": context["n"],
        "k": context["k"],
        "q_value": Q_VALUE,
        "risk_free": RISK_FREE,
        "tickers": tuple(tickers),
        "returns": tuple(float(v) for v in mu),
        "covariance_flat": tuple(float(v) for v in sigma.reshape(-1)),
        "ising_pauli_labels": tuple(labels),
        "ising_coefficients": tuple(coeffs),
        "ising_offset": float(context["offset"]),
        "exact_energy": float(context["exact_energy"]),
        "exact_bitstrings": tuple(context["exact_bitstrings"]),
        "selected_assets": tuple(classical["selected_assets"]),
        "n_parameters": int(context["n_parameters"]),
        "split_group": context["problem_id"],
    })

    local_pairs = classical["pair_table"].copy()
    local_pairs.insert(0, "problem_id", context["problem_id"])
    local_pairs.insert(1, "hamiltonian_id", context["hamiltonian_id"])
    local_pairs.insert(2, "n", context["n"])
    local_pairs.insert(3, "k", context["k"])
    pair_frames.append(local_pairs)

    local_parameters = context["parameter_map"].copy()
    local_parameters.insert(0, "problem_id", context["problem_id"])
    local_parameters.insert(1, "hamiltonian_id", context["hamiltonian_id"])
    parameter_frames.append(local_parameters)

    bank_df = bank_frames[k_value]
    if not bank_df.empty:
        local_theta = bank_df.copy()
        local_theta["theta_sin"] = local_theta["best_parameters"].map(
            lambda theta: tuple(float(v) for v in np.sin(np.asarray(theta, dtype=float)))
        )
        local_theta["theta_cos"] = local_theta["best_parameters"].map(
            lambda theta: tuple(float(v) for v in np.cos(np.asarray(theta, dtype=float)))
        )
        local_theta["split_group"] = local_theta["problem_id"]
        theta_frames.append(local_theta)

    for _, pair_row in local_pairs.iterrows():
        pair_qubits = {int(pair_row["asset_index_i"]), int(pair_row["asset_index_j"])}
        for _, parameter_row in local_parameters.iterrows():
            physical_qubits = set(int(v) for v in parameter_row["physical_qubits"])
            intersection = pair_qubits.intersection(physical_qubits)
            if not intersection:
                continue
            link_frames.append(pd.DataFrame([{
                "problem_id": context["problem_id"],
                "hamiltonian_id": context["hamiltonian_id"],
                "n": context["n"],
                "k": context["k"],
                "pair": pair_row["pair"],
                "asset_index_i": int(pair_row["asset_index_i"]),
                "asset_index_j": int(pair_row["asset_index_j"]),
                "G_ij": float(pair_row["G_ij"]),
                "G_ij_relative": float(pair_row["G_ij_relative"]),
                "theta_index": int(parameter_row["theta_index"]),
                "ansatz_gate_type": parameter_row["ansatz_gate_type"],
                "physical_qubits": tuple(parameter_row["physical_qubits"]),
                "touches_any_pair_qubit": True,
                "touches_both_pair_qubits": bool(pair_qubits.issubset(physical_qubits)),
                "n_pair_qubits_touched": int(len(intersection)),
                "angular_period": float(parameter_row["angular_period"]),
                "periodicity_structural_class": parameter_row["periodicity_structural_class"],
            }]))

problem_bank_df = pd.DataFrame(problem_rows).sort_values("k").reset_index(drop=True)
pair_bank_df = pd.concat(pair_frames, ignore_index=True)
parameter_bank_df = pd.concat(parameter_frames, ignore_index=True)
theta_bank_df = pd.concat(theta_frames, ignore_index=True) if theta_frames else pd.DataFrame()
pair_theta_link_df = pd.concat(link_frames, ignore_index=True) if link_frames else pd.DataFrame()

print("problemas:", len(problem_bank_df))
print("pares:", len(pair_bank_df))
print("parâmetros estruturais:", len(parameter_bank_df))
print("execuções theta disponíveis:", len(theta_bank_df))
print("ligações par-theta:", len(pair_theta_link_df))
display(problem_bank_df[["problem_id", "k", "exact_energy", "n_parameters", "selected_assets"]])
display(pair_theta_link_df.head(20))


## 15. Leitura correta para o Transformer

A versão 19.5 produz o **esquema** de treinamento, mas não declara generalização com apenas uma janela financeira.

### Entradas recomendadas

- coeficientes do Hamiltoniano ou, de forma equivalente, retorno e covariância;
- cardinalidade $k$;
- mapa estrutural do ansatz;
- $G_{ij}$ e penalidades direcionais como features clássicas auxiliares.

### Alvos recomendados

- energia e gap exatos;
- $P(x^*)$;
- bitstring ótimo ou distribuição de bitstrings;
- $\sin\theta$ e $\cos\theta$, nunca MSE cru como único alvo.

### Separação dos dados

Todos os vetores do mesmo `problem_id` pertencem ao mesmo grupo. Para avaliar generalização real serão necessários múltiplos Hamiltonianos, obtidos por diferentes janelas temporais e valores de $q$.


In [ ]:
# ============================================================
# 16. SALVAR TABELAS, AUDITORIAS E MANIFESTO
# ============================================================


def csv_safe(df):
    out = df.copy()
    for column in out.columns:
        if out.empty:
            break
        if out[column].map(
            lambda value: isinstance(value, (list, tuple, dict, np.ndarray))
        ).any():
            out[column] = out[column].map(
                lambda value: json.dumps(
                    value.tolist() if isinstance(value, np.ndarray) else value,
                    ensure_ascii=False,
                )
                if isinstance(value, (list, tuple, dict, np.ndarray))
                else value
            )
    return out


# Caso clássico original
csv_safe(enumeration_df).to_csv(classical_dir / "enumeration_k4.csv", index=False)
csv_safe(asset_decision_df).to_csv(classical_dir / "asset_decision_margins_k4.csv", index=False)
csv_safe(pair_table).to_csv(classical_dir / "pair_conditional_energies_k4.csv", index=False)
csv_safe(pair_category_selection_df).to_csv(classical_dir / "pair_category_selection_k4.csv", index=False)
ablation_summary_df.to_csv(classical_dir / "old_margin_ablation_k4.csv", index=False)
(classical_dir / "classical_audit_k4.json").write_text(
    json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8"
)

# Tabelas multi-k e quânticas
csv_safe(problem_bank_df).to_csv(transformer_dir / "problem_hamiltonian_bank.csv", index=False)
csv_safe(pair_bank_df).to_csv(transformer_dir / "pair_gap_bank.csv", index=False)
csv_safe(parameter_bank_df).to_csv(transformer_dir / "parameter_structure_bank.csv", index=False)
csv_safe(pair_theta_link_df).to_csv(transformer_dir / "pair_theta_link_bank.csv", index=False)
if not theta_bank_df.empty:
    theta_bank_df.to_pickle(transformer_dir / "theta_execution_bank.pkl")
    csv_safe(theta_bank_df.drop(columns=["counts"], errors="ignore")).to_csv(
        transformer_dir / "theta_execution_bank.csv", index=False
    )

# Experimento de máscaras e cronogramas 7 -> 30 -> 7
if not staged_mask_runs_df.empty:
    staged_mask_runs_df.to_pickle(staged_dir / "staged_mask_execution_bank.pkl")
    csv_safe(
        staged_mask_runs_df.drop(
            columns=[
                "initial_theta",
                "common_initial_theta",
                "best_parameters",
                "stage_records",
                "objective_history",
                "best_estimator_gap_history",
                "counts_shots",
            ],
            errors="ignore",
        )
    ).to_csv(staged_dir / "staged_mask_execution_bank.csv", index=False)
    csv_safe(staged_mask_summary_df).to_csv(
        staged_dir / "staged_mask_summary.csv", index=False
    )
    csv_safe(
        staged_mask_stage_df.drop(
            columns=["theta_start", "theta_final"], errors="ignore"
        )
    ).to_csv(staged_dir / "staged_mask_stage_rows.csv", index=False)
    csv_safe(staged_mask_stage_summary_df).to_csv(
        staged_dir / "staged_mask_stage_summary.csv", index=False
    )
    csv_safe(staged_mask_paired_wide_df).to_csv(
        staged_dir / "staged_mask_paired_wide.csv", index=False
    )
    csv_safe(staged_mask_tests_df).to_csv(
        staged_dir / "staged_mask_paired_tests.csv", index=False
    )
    csv_safe(staged_mask_convergence_df).to_csv(
        staged_dir / "staged_mask_convergence.csv", index=False
    )

(staged_dir / "staged_mask_metadata.json").write_text(
    json.dumps(staged_mask_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


for name, dataframe in {
    "problem_hamiltonian_bank": problem_bank_df,
    "pair_gap_bank": pair_bank_df,
    "parameter_structure_bank": parameter_bank_df,
    "pair_theta_link_bank": pair_theta_link_df,
}.items():
    try:
        dataframe.to_parquet(transformer_dir / f"{name}.parquet", index=False)
    except Exception as exc:
        warnings.warn(f"{name}.parquet não foi salvo: {type(exc).__name__}: {exc}")

n_complete_problems = int(bank_status_df["complete"].sum())
readiness = {
    "schema_ready": True,
    "theta_banks_complete": bool(bank_status_df["complete"].all()),
    "n_problem_ids": int(len(problem_bank_df)),
    "n_distinct_hamiltonians": int(problem_bank_df["hamiltonian_id"].nunique()),
    "architecture_experiments_ready": bool(n_complete_problems > 0),
    "staged_mask_experiment_enabled": bool(RUN_STAGED_MASK_EXPERIMENT),
    "staged_mask_experiment_executed": bool(
        staged_mask_metadata.get("executed", False)
    ),
    "staged_mask_rows": int(len(staged_mask_runs_df)),
    "staged_mask_complete_restarts": int(
        staged_mask_runs_df["restart"].nunique() if not staged_mask_runs_df.empty else 0
    ),
    "staged_distribution_shots": int(STAGED_DISTRIBUTION_SHOTS),
    "generalization_claim_ready": bool(
        problem_bank_df["hamiltonian_id"].nunique() >= 8
    ),
    "required_next_step": (
        "gerar múltiplas janelas temporais/q antes de alegar generalização"
        if problem_bank_df["hamiltonian_id"].nunique() < 8
        else "executar split por Hamiltoniano"
    ),
}

manifest = {
    **CONFIG,
    "stock_path": str(stock_path.resolve()),
    "data_hash": DATA_HASH,
    "output_dir": str(output_dir.resolve()),
    "classical_audit": audit,
    "periodicity_audit_k4": periodicity_audit,
    "cardinality_audit": cardinality_audit_df.to_dict("records"),
    "bank_status": bank_status_df.to_dict("records"),
    "staged_mask_experiment": staged_mask_metadata,
    "dataset_readiness": readiness,
}
(output_dir / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(json.dumps(readiness, indent=2, ensure_ascii=False))
print("arquivos salvos em:", output_dir.resolve())


# Parte VIII — interpretação científica

A versão 19.5.5 não pressupõe que um cronograma será superior. Ela separa cinco perguntas.

## 1. Os 23 parâmetros podem ser constantes?

Compare:

- `active7_random`;
- `active7_zero`;
- `active7_anchor`.

Se apenas a âncora funcionar, os sete são suficientes condicionalmente a um fundo estrutural correto. Se zero e âncora funcionarem, uma configuração universal simples pode existir. Se todos falharem, os 23 precisam participar da navegação.

## 2. O aquecimento inicial com sete ajuda?

Compare:

- `hybrid_7x30_30to150_7to250`;
- `hybrid_30to150_7to250`.

A diferença mede o valor das primeiras 30 avaliações restritas aos sete.

## 3. Vinte ou trinta avaliações são melhores?

Compare diretamente:

- `hybrid_7x20_30to150_7to250`;
- `hybrid_7x30_30to150_7to250`.

O resultado deve considerar qualidade final, taxa de alcance dos gaps e custo real, não apenas a curva inicial.

## 4. Voltar aos sete no final ajuda?

Compare:

- `hybrid_7x30_30to150_7to250`;
- `hybrid_7x30_30to250`.

Se o primeiro preservar a qualidade com menor variabilidade, há evidência de que a geometria se concentra novamente nos sete perto do ótimo. Se o segundo for melhor, a fase completa ainda precisa continuar além da avaliação 150.

## 5. O cronograma supera o baseline?

Todos os regimes são comparados de forma pareada com `full_30`. A hipótese forte exige simultaneamente:

\[
P(x^*)_{\mathrm{híbrido}} \approx P(x^*)_{30},
\]

\[
\Delta E_{\mathrm{híbrido}} \approx \Delta E_{30},
\]

com menor tempo, menor variabilidade ou maior taxa de sucesso precoce.

A tabela de etapas registra $P(x^*)$ e o gap exato no início e no fim de cada fase. Isso permite verificar diretamente onde ocorre a melhora: na fase de sete, na fase de 30 ou no refinamento final.


# Guia de execução

1. Execute a reconstrução clássica, a codificação Ising e a construção dos contextos quânticos.
2. Os bancos multi-$k$ continuam opcionais; use `GENERATE_OR_RESUME_BANKS=True` somente quando também quiser atualizá-los.
3. Para executar o novo experimento, defina:

```python
RUN_STAGED_MASK_EXPERIMENT = True
```

4. O desenho padrão usa 100 restarts e oito regimes por restart. Os checkpoints são salvos após cada combinação `restart × regime`.
5. A âncora é procurada nos bancos existentes. Como você já executou a comparação anterior, o notebook deve reutilizar automaticamente o melhor resultado `full_30` salvo em `05_active7_vs_full30_experiment`.
6. Caso nenhum banco exista, cinco restarts `full_30` independentes são executados apenas para construir a âncora.
7. O cronograma principal executa:

```text
7 ativos: 20 ou 30 avaliações
30 parâmetros: até o orçamento acumulado 150
7 ativos: orçamento final até 250
```

8. Cada etapa recebe o vetor final completo da etapa anterior. Ao voltar aos sete, os outros 23 ficam congelados exatamente nos valores encontrados ao fim da fase de 30.
9. `maxiter` do COBYLA é tratado como orçamento máximo de avaliações. Uma etapa pode encerrar antes se o próprio COBYLA declarar convergência; por isso o notebook registra orçamento solicitado e `nfev` efetivamente usado.
10. Examine primeiro:
   - `staged_mask_summary.csv`;
   - `staged_mask_stage_summary.csv`;
   - `staged_mask_paired_tests.csv`;
   - `staged_mask_convergence.csv`.
11. Os gráficos principais são:
   - `staged_masks_background_convergence.png`;
   - `staged_masks_schedule_convergence.png`;
   - `staged_masks_probability.png`;
   - `staged_masks_exact_gap.png`;
   - `staged_masks_cost_quality.png`.
12. Não conclua eficiência apenas por `nfev`. Compare também $P(x^*)$, gap exato, taxa de sucesso, distribuição com 4096 shots e variabilidade entre restarts.
